In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
from google.colab import files
uploaded=files.upload()


In [ ]:
import io
df= pd.read_excel(io.BytesIO(uploaded['merged_90_c1 (1).xlsx']))

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df = df.drop(['patient_id'], axis=1)

In [ ]:
train_df.columns

In [ ]:
train_df=train_df.drop('Dataset',1)

In [ ]:
from sklearn.feature_selection import chi2
import pandas as pd

# Assuming train_df is your DataFrame and the target column is named 'target'

# Sample Data (Replace this with your actual DataFrame)
# train_df = pd.DataFrame({
#     'feature1': [0, 1, 0, 1, 1],
#     'feature2': [1, 1, 1, 0, 0],
#     'feature3': [0, 0, 1, 1, 0],
#     'target': [0, 1, 1, 0, 1]
# })

# Separate the features and the target
X = df.drop('dialysis', axis=1)
y = df['dialysis']

# Calculate the Chi-Squared statistic
chi_scores, p_values = chi2(X, y)

# Create a DataFrame to display the results
chi_squared_results = pd.DataFrame({
    'Feature': X.columns,
    'Chi-Squared': chi_scores,
    'P-Value': p_values
})

chi_squared_results.sort_values(by='Chi-Squared', ascending=False)


In [ ]:
from sklearn.feature_selection import mutual_info_classif
import pandas as pd

# Assuming train_df is your DataFrame and the target column is named 'target'

# Separate the features and the target
X = df.drop('dialysis', axis=1)
y = df['dialysis']

# Calculate Mutual Information
mutual_info = mutual_info_classif(X, y)

# Create a DataFrame to display the results
mutual_info_results = pd.DataFrame({
    'Feature': X.columns,
    'Mutual Information': mutual_info
})

mutual_info_results.sort_values(by='Mutual Information', ascending=False)


In [ ]:
from sklearn.linear_model import LogisticRegression
import pandas as pd

# Assuming train_df is your DataFrame and the target column is named 'target'

# Separate the features and the target
X = df.drop('dialysis', axis=1)
y = df['dialysis']

# Create a logistic regression model with L1 regularization (Lasso)
model = LogisticRegression(penalty='l1', solver='liblinear')

# Fit the model
model.fit(X, y)

# Extract the coefficients
feature_importance = pd.Series(model.coef_[0], index=X.columns)

# Create a DataFrame to display the results
lasso_results = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': feature_importance
})

lasso_results.sort_values(by='Coefficient', ascending=False)


In [ ]:
from matplotlib import pyplot as plt
_df_0['Chi-Squared'].plot(kind='hist', bins=20, title='Chi-Squared')
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
from sklearn.decomposition import PCA
import plotly.express as px

# Assuming df is your dataframe with binary features and 'target' is your binary target.
#df = df.drop(['patient_id'], axis=1)

# Apply PCA and keep the first two principal components
pca = PCA(n_components=2)
components = pca.fit_transform(df.drop('dialysis', axis=1))

# Create a new DataFrame with the principal components
pca_df = pd.DataFrame(components, columns=['PC1', 'PC2'])
pca_df['dialysis'] = df['dialysis']  # Add the target back in

# Now create a parallel coordinates plot with the principal components
# Use a continuous color scale with only two colors for the binary target
fig = px.parallel_coordinates(
    pca_df,
    color='dialysis',
    dimensions=['PC1', 'PC2'],
    color_continuous_scale=[(0, 'blue'), (1, 'red')]  # Define the two colors for 0 and 1
)

# To ensure that only 0 and 1 are shown in the legend with discrete colors
fig.update_traces(line=dict(colorscale=[(0, 'blue'), (1, 'red')], cmin=0, cmax=1))

fig.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming pca_df is already created with PCA components and 'dialysis' column

# Create a violin plot for each PCA component
plt.figure(figsize=(12, 6))

# Violin plot for PC1
plt.subplot(1, 2, 1)
sns.violinplot(x='dialysis', y='PC1', data=pca_df)
plt.title('Violin plot of PC1')

# Violin plot for PC2
plt.subplot(1, 2, 2)
sns.violinplot(x='dialysis', y='PC2', data=pca_df)
plt.title('Violin plot of PC2')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Function to plot stacked bar chart for each feature
def plot_stacked_bar_chart(df, feature):
    counts = df.groupby([feature, 'dialysis']).size().unstack()
    counts_norm = counts.div(counts.sum(axis=1), axis=0)
    counts_norm.plot(kind='bar', stacked=True)
    plt.title(f'Stacked Bar Chart for {feature}')
    plt.xlabel(feature)
    plt.ylabel('Proportion')
    plt.show()

# Plotting for a few features due to high number of features
for feature in df.columns[:10]:  # Adjust the range as needed
    plot_stacked_bar_chart(df, feature)


In [ ]:
import plotly.express as px

# Selecting a subset of features for better visualization
selected_features = df.columns[:10].tolist()  # You can adjust the number of features here
selected_features.append('dialysis')

# Create the parallel coordinates plot
fig = px.parallel_coordinates(df[selected_features], color='dialysis',
                              labels={col: col for col in selected_features},
                              color_continuous_scale=px.colors.diverging.Tealrose,
                              color_continuous_midpoint=0.5)
fig.show()


In [ ]:
from sklearn.manifold import TSNE
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Assuming 'df' is your DataFrame and it contains features and a binary target column named 'dialysis'.
# We'll take a random sample of 1000 rows from the dataframe for t-SNE.


# Standardize the features
features = df.drop('dialysis', axis=1)
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# Apply t-SNE to reduce dimensions to 2D space for visualization
tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(features_scaled)

# Create a DataFrame with t-SNE components
tsne_df = pd.DataFrame(X_tsne, columns=['t-SNE 1', 't-SNE 2'])
tsne_df['dialysis'] = df['dialysis'].values

# Plot t-SNE components with class labels
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='t-SNE 1', y='t-SNE 2',
    hue='dialysis',
    palette={0: 'blue', 1: 'red'},
    data=tsne_df,
    legend='full',
    alpha=0.7
)
#plt.title('t-SNE plot of Dialysis Classes')
plt.savefig("t-SNE plot of Dialysis Classes.pdf", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go

# Assuming 'df' is your DataFrame and 'dialysis' is your target variable.

# Standardize the features (important for PCA)
features = df.drop('dialysis', axis=1)
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# Apply PCA and keep the first three principal components
pca = PCA(n_components=3)
components = pca.fit_transform(features_scaled)

# Create a DataFrame with the principal components
pca_df = pd.DataFrame(data=components, columns=['PC1', 'PC2', 'PC3'])
pca_df['dialysis'] = df['dialysis'].values  # Add the target variable back in

# Create traces for each class for the 3D scatter plot
trace0 = go.Scatter3d(
    x=pca_df[pca_df['dialysis'] == 0]['PC1'],
    y=pca_df[pca_df['dialysis'] == 0]['PC2'],
    z=pca_df[pca_df['dialysis'] == 0]['PC3'],
    mode='markers',
    marker=dict(
        size=3,
        color='blue',  # Class 0 points will be blue
        opacity=0.8
    ),
    name='Class 0'
)

trace1 = go.Scatter3d(
    x=pca_df[pca_df['dialysis'] == 1]['PC1'],
    y=pca_df[pca_df['dialysis'] == 1]['PC2'],
    z=pca_df[pca_df['dialysis'] == 1]['PC3'],
    mode='markers',
    marker=dict(
        size=3,
        color='red',  # Class 1 points will be red
        opacity=0.8
    ),
    name='Class 1'
)

# Create the figure
fig = go.Figure(data=[trace0, trace1])

# Update the layout of the plot
fig.update_layout(
    title='3D PCA Scatter Plot',
    autosize=False,
    width=800,
    height=800,
    margin=dict(l=65, r=50, b=65, t=90),
    scene=dict(
        xaxis_title='PC1',
        yaxis_title='PC2',
        zaxis_title='PC3',
        xaxis=dict(backgroundcolor="rgb(200, 200, 230)"),
        yaxis=dict(backgroundcolor="rgb(230, 200,230)"),
        zaxis=dict(backgroundcolor="rgb(230, 230,200)")
    )
)

# Show the figure
fig.show()


In [ ]:
# Identify binary columns
binary_cols = [col for col in df.columns if df[col].nunique() == 2 and 1 in df[col].unique()]

# Calculate mean and sort
mean_values = df[binary_cols].mean().sort_values(ascending=False)

# Display top 10 sorted means
print(mean_values.head(10))

# Visualization for top 10
plt.figure(figsize=(10, 6))
mean_values.head(10).plot(kind='bar')
#plt.title('Top 10 Binary Features Based on Mean')
plt.ylabel('Mean Value (Proportion of 1s)')
plt.xlabel('Binary Features')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Identify binary columns
binary_cols = [col for col in df.columns if df[col].nunique() == 2 and 1 in df[col].unique()]

# Calculate mean and sort
mean_values = df[binary_cols].mean().sort_values(ascending=False)

# Combine top 5 and least 5
top_5 = mean_values.head(5)
least_5 = mean_values.tail(5)
combined = pd.concat([top_5, least_5])

# Display
print(combined)

# Visualization
plt.figure(figsize=(10, 5))
combined.plot(kind='bar', color=['blue']*5 + ['red']*5)  # Different colors for top and least
#plt.title('Top 5 and Least 5 Binary Features Based on Mean')
plt.ylabel('Mean Value (Proportion of 1s)')
plt.xlabel('Binary Features')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Identify binary columns
binary_cols = [col for col in df.columns if df[col].nunique() == 2 and 1 in df[col].unique() and col != 'dialysis']

# Calculate mean and sort
mean_values = df[binary_cols].mean().sort_values(ascending=False)

# Combine top 5 and least 5
top_5 = mean_values.head(5)
least_5 = mean_values.tail(5)
combined = pd.concat([top_5, least_5])

# Display
print(combined)

# Visualization
plt.figure(figsize=(10, 6))
combined.plot(kind='bar', color=['blue']*5 + ['red']*5)  # Different colors for top and least
#plt.title('Top 5 and Least 5 Binary Features Based on Mean (Excluding Dialysis)')
plt.ylabel('Mean Value (Proportion of 1s)')
plt.xlabel('Binary Features')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Setting a seaborn style
sns.set_style("whitegrid")

# Enhanced bar chart
plt.figure(figsize=(10, 6))
ax = sns.barplot(x=combined.index, y=combined.values, palette=['blue']*5 + ['red']*5)
#plt.title('Top 5 and Least 5 Binary Features Based on Mean (Excluding Dialysis)', fontsize=16)
plt.ylabel('Mean Value (Proportion of 1s)', fontsize=14)
plt.xlabel('Binary Features', fontsize=14)
plt.xticks(rotation=45, fontsize=12)

# Annotate bars with their values
for p in ax.patches:
    ax.annotate(f"{p.get_height():.3f}",
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center',
                fontsize=12, color='black',
                xytext=(0, 10),
                textcoords='offset points')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.pointplot(x=combined.index, y=combined.values, join=False, palette=['blue']*5 + ['red']*5)
#plt.title('Top 5 and Least 5 Binary Features Based on Mean (Excluding Dialysis)', fontsize=16)
plt.ylabel('Mean Value (Proportion of 1s)', fontsize=14)
plt.xlabel('Binary Features', fontsize=14)
plt.xticks(rotation=45, fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Assuming 'combined' is a Series with the top and least 5 features' means
# and 'df' is the original DataFrame including the 'dialysis' target.

import matplotlib.pyplot as plt

# Data preparation
top_least_counts = df[combined.index].apply(pd.Series.value_counts).T
top_least_counts['total'] = top_least_counts.sum(axis=1)
top_least_props = top_least_counts.div(top_least_counts['total'], axis=0).drop(columns='total')

# Plot
ax = top_least_props.plot(kind='bar', stacked=True, color=['blue', 'orange'], figsize=(10, 6))
ax.set_ylabel('Proportion of Values')
ax.set_xlabel('Features')
ax.set_title('Proportion of 0s and 1s in Top and Least 5 Features')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
df_without_dialysis = df.drop(columns=['dialysis'])

# Count the number of 1s
count_1s = df_without_dialysis[df_without_dialysis == 1].count().sum()
print(count_1s)

In [ ]:
df['dialysis'].value_counts()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.svm import SVC  # Support Vector Machine
from sklearn.tree import DecisionTreeClassifier  # Decision Tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score


In [ ]:
import pandas as pd
from pylab import rcParams
import seaborn as sb
from scipy.stats.stats import kendalltau

# ***Real Data***

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LeakyReLU
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = df.drop('dialysis', axis=1)
# Replace 'dialysis' with the actual target column name
y = df['dialysis']
# Drop the 'patient-id' column from X before splitting


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


X_test.columns = X_train.columns

#classifier = LogisticRegression()
classifier = RandomForestClassifier(random_state=42)
classifier.fit(X_train, y_train)

# Step 8: Evaluate the trained model
y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)


print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy}")

In [ ]:
classifier = LogisticRegression()

classifier.fit(X_train, y_train)

# Step 8: Evaluate the trained model
y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)


print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy}")

In [ ]:
classifier_svm = SVC(random_state=42)
classifier_svm.fit(X_train, y_train)

# Evaluate SVM
y_pred_svm = classifier_svm.predict(X_test)
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print("\nSupport Vector Machine:")
print(classification_report(y_test, y_pred_svm))
print(f"Accuracy: {accuracy_svm}")

# Create and fit Decision Tree classifier
classifier_dt = DecisionTreeClassifier(random_state=42)
classifier_dt.fit(X_train, y_train)

# Evaluate Decision Tree
y_pred_dt = classifier_dt.predict(X_test)
accuracy_dt = accuracy_score(y_test, y_pred_dt)
print("\nDecision Tree:")
print(classification_report(y_test, y_pred_dt))
print(f"Accuracy: {accuracy_dt}")

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Assuming 'classifier', 'X_test', and 'y_test' are defined
ConfusionMatrixDisplay.from_estimator(classifier_dt, X_test, y_test)

# Remove lines
plt.gca().xaxis.grid(False)  # Remove vertical lines/grid
plt.gca().yaxis.grid(False)  # Remove horizontal lines/grid

plt.show()

In [ ]:
# First, concatenate X_train and y_train along the columns
train_df = pd.concat([X_train, y_train], axis=1)

print(train_df)


# **Copula2**

In [ ]:
minority_data = train_df[train_df['dialysis'] == 1].drop(columns=['dialysis'])
corr_kendall = minority_data.corr(method='kendall')


corr_pearson = np.sin(np.pi / 2 * corr_kendall)

In [ ]:
import numpy as np
from scipy.stats import gaussian_kde
from scipy.interpolate import interp1d

def kernel_cdf(x, probs=None):

    if probs is not None:
        x_vals = np.quantile(x, probs)
        kde = gaussian_kde(x_vals)
    else:
        kde = gaussian_kde(x)

    # Define a range of values for which to calculate the CDF
    x_eval = np.linspace(x.min(), x.max(), 512)
    delta = x_eval[1] - x_eval[0]

    # Calculate the cumulative density function
    cdf = np.cumsum(kde(x_eval) * delta)

    # Normalize the CDF to ensure it goes to 1
    cdf /= cdf[-1]

    # Create an interpolation function for the CDF
    cdf_function = interp1d(x_eval, cdf, kind='linear', bounds_error=False, fill_value=(0, 1))

    return cdf_function



In [ ]:
cdf_functions = {col: kernel_cdf(minority_data[col]) for col in minority_data.columns}


In [ ]:
pip install copulas

In [ ]:
from copulas.multivariate import GaussianMultivariate
from copulas.univariate import UniformUnivariate
import numpy as np


linear_corr =corr_pearson
n_columns = linear_corr.shape[0]  # Number of dimensions
n_samples = 2407  #

# Define the Gaussian copula with the given correlation matrix
copula = GaussianMultivariate(distribution=UniformUnivariate())


# Fit the copula model to the empirical data 'df'
copula.fit(minority_data)
copula.covariance = linear_corr  # Set the copula correlation

# Sample from the copula
c_samples = copula.sample(n_samples)

print(c_samples)


In [ ]:
c_samples=pd.DataFrame(c_samples)

# **CTGAN**

In [ ]:
pip install ctgan

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
import pandas as pd
from ctgan import CTGAN

# Initial setup
df20 = train_df[train_df['dialysis']==1]
focused_columns = train_df.columns
ctgan = CTGAN(epochs=38)
# Lists to store results for each classifier
metrics = ['recall', 'precision', 'f1-score', 'accuracy']
classifiers = ['DecisionTree', 'RandomForest', 'SVM', 'LogisticRegression']
results = {clf: {metric: [] for metric in metrics} for clf in classifiers}
# Dictionary to store recall scores of class 1 for each classifier
recall_scores_class_1 = {
    'DecisionTree': [],
    'RandomForest': [],
    'SVM': [],
    'LogisticRegression': []
}

# Run the process 25 times
for _ in range(25):
    # Train CTGAN model on transformed copula samples
    ctgan.fit(df20, focused_columns)

    # Create synthetic data
    synthetic_data = ctgan.sample(2407)

    # Concatenate the DataFrames
    df30 = pd.concat([train_df, synthetic_data], ignore_index=True)

    # Data for classifiers
    X = df30.drop('dialysis', axis=1)
    y = df30['dialysis']

    # Initialize classifiers
    classifiers = {
        'DecisionTree': DecisionTreeClassifier(random_state=42),
        'RandomForest': RandomForestClassifier(random_state=42),
        'SVM': SVC(random_state=42),
        'LogisticRegression': LogisticRegression(random_state=42)
    }

    for name, clf in classifiers.items():
        # Fit classifier
        clf.fit(X, y)

        # Evaluate classifier
        y_pred = clf.predict(X_test)
        report = classification_report(y_test, y_pred, output_dict=True)
        # Store metrics
        results[name]['recall'].append(report['weighted avg']['recall'])
        results[name]['precision'].append(report['weighted avg']['precision'])
        results[name]['f1-score'].append(report['weighted avg']['f1-score'])
        results[name]['accuracy'].append(accuracy_score(y_test, y_pred))
        # Store recall score of class 1
        recall_scores_class_1[name].append(report['1']['recall'])

# Now, recall_scores_class_1 dictionary contains the recall scores of class 1 for all models


In [ ]:
import matplotlib.pyplot as plt

# Plotting the recall scores for class 1
fig, ax = plt.subplots(figsize=(10, 6))

# Data for plotting
data = [recall_scores_class_1['DecisionTree'],
        recall_scores_class_1['RandomForest'],
        recall_scores_class_1['SVM'],
        recall_scores_class_1['LogisticRegression']]

# Creating the boxplot
ax.boxplot(data)

# Setting the labels for each boxplot
ax.set_xticklabels(['Decision Tree', 'Random Forest', 'SVM', 'Logistic Regression'])
#ax.set_title('Recall Scores of Class 1 for Each Classifier')
ax.set_ylabel('Recall Score')
ax.set_xlabel('Classifiers')
#plt.savefig("recall_classs1_GAN.pdf", dpi=300, bbox_inches='tight')


# Show the plot
plt.show()


In [ ]:
recall_scores_class_1={'DecisionTree': [0.575,
  0.65,
  0.65,
  0.575,
  0.575,
  0.575,
  0.6,
  0.625,
  0.6,
  0.625,
  0.625,
  0.525,
  0.55,
  0.675,
  0.575,
  0.55,
  0.625,
  0.625,
  0.525,
  0.6,
  0.525,
  0.575,
  0.6,
  0.575,
  0.55],
 'RandomForest': [0.6,
  0.55,
  0.575,
  0.6,
  0.575,
  0.575,
  0.6,
  0.625,
  0.625,
  0.575,
  0.575,
  0.6,
  0.625,
  0.625,
  0.6,
  0.575,
  0.625,
  0.525,
  0.6,
  0.6,
  0.575,
  0.625,
  0.65,
  0.575,
  0.55],
 'SVM': [0.65,
  0.6,
  0.65,
  0.625,
  0.625,
  0.65,
  0.625,
  0.625,
  0.65,
  0.6,
  0.625,
  0.65,
  0.625,
  0.65,
  0.65,
  0.65,
  0.65,
  0.625,
  0.65,
  0.65,
  0.625,
  0.625,
  0.65,
  0.6,
  0.625],
 'LogisticRegression': [0.7,
  0.6,
  0.65,
  0.625,
  0.65,
  0.65,
  0.65,
  0.65,
  0.65,
  0.625,
  0.65,
  0.65,
  0.65,
  0.65,
  0.675,
  0.65,
  0.75,
  0.65,
  0.625,
  0.675,
  0.65,
  0.625,
  0.65,
  0.65,
  0.65]}


In [ ]:
fig, axes = plt.subplots(len(metrics), len(classifiers), figsize=(20, 15))
for i, metric in enumerate(metrics):
    for j, clf in enumerate(classifiers):
        axes[i, j].boxplot(results[clf][metric])
        axes[i, j].set_title(f'{metric} of {clf}')
        axes[i, j].set_ylabel(metric.capitalize())
plt.tight_layout()
plt.show()

In [ ]:
pip install table_evaluator

In [ ]:
from table_evaluator import load_data, TableEvaluator

In [ ]:
from google.colab import files
uploaded=files.upload()


In [ ]:
import io
synthetic_data= pd.read_csv(io.BytesIO(uploaded['synthetic_data_gan.csv']))

In [ ]:
synthetic_data

In [ ]:
synthetic_data.to_csv("synthetic_data_gan.csv", index=False)

In [ ]:
table_evaluator = TableEvaluator(train_df, synthetic_data)

In [ ]:
table_evaluator.visual_evaluation()

In [ ]:
from google.colab import files
uploaded=files.upload()


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import io
df= pd.read_excel(io.BytesIO(uploaded['merged_90_c1.xlsx']))

In [ ]:
from google.colab import files
uploaded=files.upload()


In [ ]:
import io
df3= pd.read_csv(io.BytesIO(uploaded['synthetic_data_copula.csv']))

In [ ]:
from google.colab import files
uploaded=files.upload()

In [ ]:
import io
df4= pd.read_csv(io.BytesIO(uploaded['synthetic_data_binary_copula.csv']))

In [ ]:
from google.colab import files
uploaded=files.upload()


In [ ]:
import io
df2= pd.read_csv(io.BytesIO(uploaded['synthetic_data_gan.csv']))

In [ ]:
from google.colab import files
uploaded=files.upload()


In [ ]:
import io
df1= pd.read_csv(io.BytesIO(uploaded['synthetic_data_smote.csv']))

In [ ]:
df10=df[df['dialysis']==1]

In [ ]:
from google.colab import files
uploaded=files.upload()

In [ ]:
import io
df3= pd.read_csv(io.BytesIO(uploaded['synthetic_data_copula.csv']))

In [ ]:
df['1.columns']

In [ ]:

df1= df1.drop(['patient_id'], axis=1)

In [ ]:
df10.shape

In [ ]:
df= df.drop(['patient_id'], axis=1)

In [ ]:
from matplotlib import pyplot as plt
results_df['P-Value'].plot(kind='hist', bins=20, title='P-Value')
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
from matplotlib import pyplot as plt
results_df['P-Value'].plot(kind='line', figsize=(8, 4), title='P-Value')
plt.gca().spines[['top', 'right']].set_visible(False)

In [ ]:
from matplotlib import pyplot as plt
results_df['Z-Statistic'].plot(kind='line', figsize=(8, 4), title='Z-Statistic')
plt.gca().spines[['top', 'right']].set_visible(False)

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
results_df.groupby('Significant').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

# ***BINOMIAL TEST***

In [ ]:
from statsmodels.stats.proportion import proportions_ztest
def compare_features_proportion_with_power_and_diff_confint(df, df1, alphas):
    """
    Perform binomial proportion tests across all matching features in two dataframes for multiple alpha levels.
    Also, calculate the power of the test and the confidence interval for the difference in proportions.

    Parameters:
    df : pandas DataFrame, first dataset
    df1 : pandas DataFrame, second dataset
    alphas : list, a list of alpha levels for the tests

    Returns:
    result_df : pandas DataFrame, containing the results for each feature at different alpha levels including power and confidence interval for the difference in proportions
    """
    results = []

    for feature in df.columns:
        count1 = df[feature].sum()
        count2 = df1[feature].sum()

        nobs1 = len(df[feature])
        nobs2 = len(df1[feature])

        prop1 = count1 / nobs1
        prop2 = count2 / nobs2

        # Standard error for the difference in two proportions
        std_error_diff = np.sqrt(prop1 * (1 - prop1) / nobs1 + prop2 * (1 - prop2) / nobs2)

        for alpha in alphas:
            # Perform the binomial proportion test
            z_stat, p_value = proportions_ztest([count1, count2], [nobs1, nobs2])

            # Determine if the result is significant based on the provided alpha
            significant = p_value < alpha

            # Calculate the confidence interval for the difference in proportions
            confint_diff = (prop1 - prop2 - std_error_diff * z_stat, prop1 - prop2 + std_error_diff * z_stat)

            # Calculate the power of the test
            effect_size = (prop1 - prop2) / std_error_diff
            #power = zt_ind_solve_power(effect_size=effect_size, nobs1=nobs1, alpha=alpha, ratio=nobs2/nobs1, alternative='two-sided')

            # Append the results
            results.append({
                'Feature': feature,
                'Alpha': alpha,
                'Z-Statistic': z_stat,
                'P-Value': p_value,
                'Significant': significant,
                'Confidence Interval for Difference': confint_diff
            })

    # Convert results to DataFrame
    result_df = pd.DataFrame(results)

    return result_df

# Example usage:
alphas = [0.01, 0.025, 0.05, 0.1, 0.2, 0.5]
results_df1= compare_features_proportion_with_power_and_diff_confint(df10, df1, alphas)
# Note: df and df1 need to be defined in your workspace.
results_df1
# will be a DataFrame containing the detailed test results for each feature at different alpha levels, including test power and confidence interval for the difference in proportions.


In [ ]:
# @title Significant

from matplotlib import pyplot as plt
import seaborn as sns
results_df1.groupby('Significant').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
# @title Z-Statistic

from matplotlib import pyplot as plt
results_df1['Z-Statistic'].plot(kind='hist', bins=20, title='Z-Statistic')
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
# @title P-Value

from matplotlib import pyplot as plt
results_df1['P-Value'].plot(kind='hist', bins=20, title='P-Value')
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
# @title Significant vs Alpha

from matplotlib import pyplot as plt
import seaborn as sns
figsize = (8,4 * len(results_df1['Significant'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(results_df1, x='Alpha', y='Significant', inner='box', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

In [ ]:
from statsmodels.stats.proportion import proportions_ztest
def compare_features_proportion_with_power_and_diff_confint(df, df1, alphas):
    """
    Perform binomial proportion tests across all matching features in two dataframes for multiple alpha levels.
    Also, calculate the power of the test and the confidence interval for the difference in proportions.

    Parameters:
    df : pandas DataFrame, first dataset
    df1 : pandas DataFrame, second dataset
    alphas : list, a list of alpha levels for the tests

    Returns:
    result_df : pandas DataFrame, containing the results for each feature at different alpha levels including power and confidence interval for the difference in proportions
    """
    results = []

    for feature in df.columns:
        count1 = df[feature].sum()
        count2 = df1[feature].sum()

        nobs1 = len(df[feature])
        nobs2 = len(df1[feature])

        prop1 = count1 / nobs1
        prop2 = count2 / nobs2

        # Standard error for the difference in two proportions
        std_error_diff = np.sqrt(prop1 * (1 - prop1) / nobs1 + prop2 * (1 - prop2) / nobs2)

        for alpha in alphas:
            # Perform the binomial proportion test
            z_stat, p_value = proportions_ztest([count1, count2], [nobs1, nobs2])

            # Determine if the result is significant based on the provided alpha
            significant = p_value < alpha

            # Calculate the confidence interval for the difference in proportions
            confint_diff = (prop1 - prop2 - std_error_diff * z_stat, prop1 - prop2 + std_error_diff * z_stat)

            # Calculate the power of the test
            effect_size = (prop1 - prop2) / std_error_diff
            #power = zt_ind_solve_power(effect_size=effect_size, nobs1=nobs1, alpha=alpha, ratio=nobs2/nobs1, alternative='two-sided')

            # Append the results
            results.append({
                'Feature': feature,
                'Alpha': alpha,
                'Z-Statistic': z_stat,
                'P-Value': p_value,
                'Significant': significant,
                'Confidence Interval for Difference': confint_diff
            })

    # Convert results to DataFrame
    result_df = pd.DataFrame(results)

    return result_df

# Example usage:
alphas = [0.01, 0.025, 0.05, 0.1, 0.2, 0.5]
results_df2= compare_features_proportion_with_power_and_diff_confint(df10, df2, alphas)
# Note: df and df1 need to be defined in your workspace.
results_df2
# will be a DataFrame containing the detailed test results for each feature at different alpha levels, including test power and confidence interval for the difference in proportions.


In [ ]:
# @title Significant vs Alpha

from matplotlib import pyplot as plt
import seaborn as sns
figsize = (8, 4* len(results_df2['Significant'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(results_df2, x='Alpha', y='Significant', inner='box', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

In [ ]:
# @title Significant

from matplotlib import pyplot as plt
import seaborn as sns
results_df2.groupby('Significant').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
# @title P-Value

from matplotlib import pyplot as plt
results_df2['P-Value'].plot(kind='hist', bins=20, title='P-Value')
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
# @title Z-Statistic

from matplotlib import pyplot as plt
results_df2['Z-Statistic'].plot(kind='hist', bins=20, title='Z-Statistic')
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
from statsmodels.stats.proportion import proportions_ztest
def compare_features_proportion_with_power_and_diff_confint(df, df1, alphas):
    """
    Perform binomial proportion tests across all matching features in two dataframes for multiple alpha levels.
    Also, calculate the power of the test and the confidence interval for the difference in proportions.

    Parameters:
    df : pandas DataFrame, first dataset
    df1 : pandas DataFrame, second dataset
    alphas : list, a list of alpha levels for the tests

    Returns:
    result_df : pandas DataFrame, containing the results for each feature at different alpha levels including power and confidence interval for the difference in proportions
    """
    results = []

    for feature in df.columns:
        count1 = df[feature].sum()
        count2 = df1[feature].sum()

        nobs1 = len(df[feature])
        nobs2 = len(df1[feature])

        prop1 = count1 / nobs1
        prop2 = count2 / nobs2

        # Standard error for the difference in two proportions
        std_error_diff = np.sqrt(prop1 * (1 - prop1) / nobs1 + prop2 * (1 - prop2) / nobs2)

        for alpha in alphas:
            # Perform the binomial proportion test
            z_stat, p_value = proportions_ztest([count1, count2], [nobs1, nobs2])

            # Determine if the result is significant based on the provided alpha
            significant = p_value < alpha

            # Calculate the confidence interval for the difference in proportions
            confint_diff = (prop1 - prop2 - std_error_diff * z_stat, prop1 - prop2 + std_error_diff * z_stat)

            # Calculate the power of the test
            effect_size = (prop1 - prop2) / std_error_diff
            #power = zt_ind_solve_power(effect_size=effect_size, nobs1=nobs1, alpha=alpha, ratio=nobs2/nobs1, alternative='two-sided')

            # Append the results
            results.append({
                'Feature': feature,
                'Alpha': alpha,
                'Z-Statistic': z_stat,
                'P-Value': p_value,
                'Significant': significant,
                'Confidence Interval for Difference': confint_diff
            })

    # Convert results to DataFrame
    result_df = pd.DataFrame(results)

    return result_df

# Example usage:
alphas = [0.01, 0.025, 0.05, 0.1, 0.2, 0.5]
results_df3= compare_features_proportion_with_power_and_diff_confint(df10, df3, alphas)
# Note: df and df1 need to be defined in your workspace.
results_df3
# will be a DataFrame containing the detailed test results for each feature at different alpha levels, including test power and confidence interval for the difference in proportions.


In [ ]:
# @title Significant vs Alpha

from matplotlib import pyplot as plt
import seaborn as sns
figsize = (8, 4 * len(results_df3['Significant'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(results_df3, x='Alpha', y='Significant', inner='box', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

In [ ]:
# @title P-Value

from matplotlib import pyplot as plt
results_df3['P-Value'].plot(kind='hist', bins=20, title='P-Value')
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
# @title Significant

from matplotlib import pyplot as plt
import seaborn as sns
results_df3.groupby('Significant').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
# @title Z-Statistic

from matplotlib import pyplot as plt
results_df3['Z-Statistic'].plot(kind='hist', bins=20, title='Z-Statistic')
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
from statsmodels.stats.proportion import proportions_ztest
def compare_features_proportion_with_power_and_diff_confint(df, df1, alphas):
    """
    Perform binomial proportion tests across all matching features in two dataframes for multiple alpha levels.
    Also, calculate the power of the test and the confidence interval for the difference in proportions.

    Parameters:
    df : pandas DataFrame, first dataset
    df1 : pandas DataFrame, second dataset
    alphas : list, a list of alpha levels for the tests

    Returns:
    result_df : pandas DataFrame, containing the results for each feature at different alpha levels including power and confidence interval for the difference in proportions
    """
    results = []

    for feature in df.columns:
        count1 = df[feature].sum()
        count2 = df1[feature].sum()

        nobs1 = len(df[feature])
        nobs2 = len(df1[feature])

        prop1 = count1 / nobs1
        prop2 = count2 / nobs2

        # Standard error for the difference in two proportions
        std_error_diff = np.sqrt(prop1 * (1 - prop1) / nobs1 + prop2 * (1 - prop2) / nobs2)

        for alpha in alphas:
            # Perform the binomial proportion test
            z_stat, p_value = proportions_ztest([count1, count2], [nobs1, nobs2])

            # Determine if the result is significant based on the provided alpha
            significant = p_value < alpha

            # Calculate the confidence interval for the difference in proportions
            confint_diff = (prop1 - prop2 - std_error_diff * z_stat, prop1 - prop2 + std_error_diff * z_stat)

            # Calculate the power of the test
            effect_size = (prop1 - prop2) / std_error_diff
            #power = zt_ind_solve_power(effect_size=effect_size, nobs1=nobs1, alpha=alpha, ratio=nobs2/nobs1, alternative='two-sided')

            # Append the results
            results.append({
                'Feature': feature,
                'Alpha': alpha,
                'Z-Statistic': z_stat,
                'P-Value': p_value,
                'Significant': significant,
                'Confidence Interval for Difference': confint_diff
            })

    # Convert results to DataFrame
    result_df = pd.DataFrame(results)

    return result_df

# Example usage:
alphas = [0.01, 0.025, 0.05, 0.1, 0.2, 0.5]
results_df4= compare_features_proportion_with_power_and_diff_confint(df10, df4, alphas)
# Note: df and df1 need to be defined in your workspace.
results_df4
# will be a DataFrame containing the detailed test results for each feature at different alpha levels, including test power and confidence interval for the difference in proportions.


In [ ]:
# @title Significant vs Alpha

from matplotlib import pyplot as plt
import seaborn as sns
figsize = (8, 4 * len(results_df4['Significant'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(results_df4, x='Alpha', y='Significant', inner='box', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

In [ ]:
# @title Z-Statistic

from matplotlib import pyplot as plt
results_df4['Z-Statistic'].plot(kind='hist', bins=20, title='Z-Statistic')
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
# @title Significant

from matplotlib import pyplot as plt
import seaborn as sns
results_df4.groupby('Significant').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
# @title P-Value

from matplotlib import pyplot as plt
results_df4['P-Value'].plot(kind='hist', bins=20, title='P-Value')
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
results_df.describe()

In [ ]:
results_df['Significant'].value_counts()

In [ ]:
from matplotlib import pyplot as plt
results_df.plot(kind='scatter', x='P-Value', y='Power', s=32, alpha=.8)
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
from matplotlib import pyplot as plt
results_df['Power'].plot(kind='hist', bins=20, title='Power')
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
from statsmodels.stats.proportion import proportions_ztest
def compare_features_proportion_with_power_and_diff_confint(df, df1, alphas):
    """
    Perform binomial proportion tests across all matching features in two dataframes for multiple alpha levels.
    Also, calculate the power of the test and the confidence interval for the difference in proportions.

    Parameters:
    df : pandas DataFrame, first dataset
    df1 : pandas DataFrame, second dataset
    alphas : list, a list of alpha levels for the tests

    Returns:
    result_df : pandas DataFrame, containing the results for each feature at different alpha levels including power and confidence interval for the difference in proportions
    """
    results = []

    for feature in df.columns:
        count1 = df[feature].sum()
        count2 = df1[feature].sum()

        nobs1 = len(df[feature])
        nobs2 = len(df1[feature])

        prop1 = count1 / nobs1
        prop2 = count2 / nobs2

        # Standard error for the difference in two proportions
        std_error_diff = np.sqrt(prop1 * (1 - prop1) / nobs1 + prop2 * (1 - prop2) / nobs2)

        for alpha in alphas:
            # Perform the binomial proportion test
            z_stat, p_value = proportions_ztest([count1, count2], [nobs1, nobs2])

            # Determine if the result is significant based on the provided alpha
            significant = p_value < alpha

            # Calculate the confidence interval for the difference in proportions
            confint_diff = (prop1 - prop2 - std_error_diff * z_stat, prop1 - prop2 + std_error_diff * z_stat)

            # Calculate the power of the test
            effect_size = (prop1 - prop2) / std_error_diff
            power = zt_ind_solve_power(effect_size=effect_size, nobs1=nobs1, alpha=alpha, ratio=nobs2/nobs1, alternative='two-sided')

            # Append the results
            results.append({
                'Feature': feature,
                'Alpha': alpha,
                'Z-Statistic': z_stat,
                'P-Value': p_value,
                'Significant': significant,
                'Confidence Interval for Difference': confint_diff,
                'Power': power
            })

    # Convert results to DataFrame
    result_df = pd.DataFrame(results)

    return result_df

# Example usage:
alphas = [0.01, 0.025, 0.05, 0.1, 0.2, 0.5]
results_df = compare_features_proportion_with_power_and_diff_confint(df10, df1, alphas)
# Note: df and df1 need to be defined in your workspace.
results_df
# will be a DataFrame containing the detailed test results for each feature at different alpha levels, including test power and confidence interval for the difference in proportions.


In [ ]:
results_df.describe()

In [ ]:
results_df['Significant'].value_counts()

In [ ]:
from matplotlib import pyplot as plt
results_df['P-Value'].plot(kind='hist', bins=20, title='P-Value')
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
results_df.groupby('Significant').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
from matplotlib import pyplot as plt
results_df['P-Value'].plot(kind='hist', bins=20, title='P-Value')
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
from matplotlib import pyplot as plt
results_df['P-Value'].plot(kind='line', figsize=(8, 4), title='P-Value')
plt.gca().spines[['top', 'right']].set_visible(False)

In [ ]:
from matplotlib import pyplot as plt
results_df['Z-Statistic'].plot(kind='line', figsize=(8, 4), title='Z-Statistic')
plt.gca().spines[['top', 'right']].set_visible(False)

In [ ]:
results_df = compare_features_proportion_with_power_and_diff_confint(df10, df_smote, alphas)
# Note: df and df1 need to be defined in your workspace.
results_df

In [ ]:
results_df.describe()

In [ ]:
results_df['Significant'].value_counts()

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
results_df.groupby('Significant').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
results_df.groupby('Significant').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
from matplotlib import pyplot as plt
results_df['P-Value'].plot(kind='hist', bins=20, title='P-Value')
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
df10=df10.drop(['patinet_id'], axis=1)

In [ ]:
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.power import NormalIndPower, zt_ind_solve_power
import numpy as np
def compare_features_proportion_with_power_and_diff_confint(df, df1, alphas):
    """
    Perform binomial proportion tests across all matching features in two dataframes for multiple alpha levels.
    Also, calculate the power of the test and the confidence interval for the difference in proportions.

    Parameters:
    df : pandas DataFrame, first dataset
    df1 : pandas DataFrame, second dataset
    alphas : list, a list of alpha levels for the tests

    Returns:
    result_df : pandas DataFrame, containing the results for each feature at different alpha levels including power and confidence interval for the difference in proportions
    """
    results = []

    for feature in df.columns:
        count1 = df[feature].sum()
        count2 = df1[feature].sum()

        nobs1 = len(df[feature])
        nobs2 = len(df1[feature])

        prop1 = count1 / nobs1
        prop2 = count2 / nobs2

        # Standard error for the difference in two proportions
        std_error_diff = np.sqrt(prop1 * (1 - prop1) / nobs1 + prop2 * (1 - prop2) / nobs2)

        for alpha in alphas:
            # Perform the binomial proportion test
            z_stat, p_value = proportions_ztest([count1, count2], [nobs1, nobs2])

            # Determine if the result is significant based on the provided alpha
            significant = p_value < alpha

            # Calculate the confidence interval for the difference in proportions
            confint_diff = (prop1 - prop2 - std_error_diff * z_stat, prop1 - prop2 + std_error_diff * z_stat)

            # Calculate the power of the test
            effect_size = (prop1 - prop2) / std_error_diff
            power = zt_ind_solve_power(effect_size=effect_size, nobs1=nobs1, alpha=alpha, ratio=nobs2/nobs1, alternative='two-sided')

            # Append the results
            results.append({
                'Feature': feature,
                'Alpha': alpha,
                'Z-Statistic': z_stat,
                'P-Value': p_value,
                'Significant': significant,
                'Confidence Interval for Difference': confint_diff,
                'Power': power
            })

    # Convert results to DataFrame
    result_df = pd.DataFrame(results)

    return result_df

# Example usage:
alphas = [0.01, 0.025, 0.05, 0.1, 0.2, 0.5]
results_df = compare_features_proportion_with_power_and_diff_confint(df10, df3, alphas)
# Note: df and df1 need to be defined in your workspace.
results_df
# will be a DataFrame containing the detailed test results for each feature at different alpha levels, including test power and confidence interval for the difference in proportions.


In [ ]:
results_df.describe()

In [ ]:
results_df['Significant'].value_counts()

In [ ]:
from matplotlib import pyplot as plt
results_df['P-Value'].plot(kind='hist', bins=20, title='P-Value')
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
results_df.groupby('Significant').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Assuming 'train_df' and 'synthetic_data' are your dataframes

# Add a column to distinguish between real and synthetic data
train_df['Type'] = 'Real'
synthetic_data['Type'] = 'Synthetic'

# Combine the datasets for easier plotting
combined_data = pd.concat([train_df, synthetic_data])

# List of features to plot (excluding the 'Type' column)
features = combined_data.drop('Type', axis=1).columns

# Create a violin plot for each feature
for feature in features:
    plt.figure(figsize=(10, 6))
    sns.violinplot(x='Type', y=feature, data=combined_data)
    #plt.title(f'{feature}')
    plt.ylabel(feature)
    plt.xlabel('Data Type')
    plt.show()


In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import pandas as pd

# Assuming you have a DataFrame 'combined_data' with a 'Type' column distinguishing real and synthetic data

# Extract features and labels for t-SNE
features = combined_data.drop('Type', axis=1).values
labels = combined_data['Type'].values

# Set the perplexity values you want to test
perplexity_values = [1,5, 30, 50, 100]

# Create subplots
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(15, 15))
axes = axes.flatten()

for i, perplexity in enumerate(perplexity_values):
    # Apply t-SNE with the current perplexity
    tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42)
    transformed_features = tsne.fit_transform(features)

    # Scatter plot
    axes[i].scatter(transformed_features[labels == 'Real', 0],
                    transformed_features[labels == 'Real', 1],
                    c='orange', label='Real Data', alpha=0.5)

    axes[i].scatter(transformed_features[labels == 'Synthetic', 0],
                    transformed_features[labels == 'Synthetic', 1],
                    c='blue', label='Synthetic Data', alpha=0.5)

    axes[i].legend()
    axes[i].set_title(f't-SNE with perplexity={perplexity}')

# Show the plot
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import pandas as pd

# Assuming you have a DataFrame 'combined_data' with a 'Type' column distinguishing real and synthetic data

# Extract features and labels for t-SNE
features = combined_data.drop('Type', axis=1).values
labels = combined_data['Type'].values

# Set the perplexity values you want to test
perplexity_values = [1,5, 30, 50, 100]

# Create subplots
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(15, 15))
axes = axes.flatten()

for i, perplexity in enumerate(perplexity_values):
    # Apply t-SNE with the current perplexity
    tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42)
    transformed_features = tsne.fit_transform(features)

    # Scatter plot
    axes[i].scatter(transformed_features[labels == 'Real', 0],
                    transformed_features[labels == 'Real', 1],
                    c='orange', label='Real Data', alpha=0.5)

    axes[i].scatter(transformed_features[labels == 'Synthetic', 0],
                    transformed_features[labels == 'Synthetic', 1],
                    c='blue', label='Synthetic Data', alpha=0.5)

    axes[i].legend()
    axes[i].set_title(f't-SNE with perplexity={perplexity}')

# Show the plot
plt.tight_layout()
plt.show()


In [ ]:
train_df

In [ ]:
train_df = pd.concat([X_train, y_train], axis=1)
train_df

# **SMOTE**

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score
from collections import Counter


# Define the resampling method
resample = SMOTE(sampling_strategy=0.8)
X = train_df.drop('dialysis', axis=1)
y = train_df['dialysis']


# Lists to store results for each classifier
metrics = ['recall', 'precision', 'f1-score', 'accuracy']
classifiers = ['DecisionTree', 'RandomForest', 'SVM', 'LogisticRegression']
results = {clf: {metric: [] for metric in metrics} for clf in classifiers}
# Dictionary to store recall scores of class 1 for each classifier
recall_scores_class_1 = {
    'DecisionTree': [],
    'RandomForest': [],
    'SVM': [],
    'LogisticRegression': []
}

# Run the process 25 times
for _ in range(25):
    # Train CTGAN model on transformed copula samples
    X = train_df.drop('dialysis', axis=1)
    y = train_df['dialysis']
    X_resampled, y_resampled = resample.fit_resample(X, y)


    # Initialize classifiers
    classifiers = {
        'DecisionTree': DecisionTreeClassifier(random_state=42),
        'RandomForest': RandomForestClassifier(random_state=42),
        'SVM': SVC(random_state=42),
        'LogisticRegression': LogisticRegression(random_state=42)
    }

    for name, clf in classifiers.items():
        # Fit classifier
        clf.fit(X_resampled, y_resampled)

        # Evaluate classifier
        y_pred = clf.predict(X_test)
        report = classification_report(y_test, y_pred, output_dict=True)
        # Store metrics
        results[name]['recall'].append(report['weighted avg']['recall'])
        results[name]['precision'].append(report['weighted avg']['precision'])
        results[name]['f1-score'].append(report['weighted avg']['f1-score'])
        results[name]['accuracy'].append(accuracy_score(y_test, y_pred))
        # Store recall score of class 1
        recall_scores_class_1[name].append(report['1']['recall'])

# Now, recall_scores_class_1 dictionary contains the recall scores of class 1 for all models


In [ ]:
import matplotlib.pyplot as plt

# Plotting the recall scores for class 1
fig, ax = plt.subplots(figsize=(10, 6))

# Data for plotting
data = [recall_scores_class_1['DecisionTree'],
        recall_scores_class_1['RandomForest'],
        recall_scores_class_1['SVM'],
        recall_scores_class_1['LogisticRegression']]

# Creating the boxplot
ax.boxplot(data)

# Setting the labels for each boxplot
ax.set_xticklabels(['Decision Tree', 'Random Forest', 'SVM', 'Logistic Regression'])
#ax.set_title('Recall Scores of Class 1 for Each Classifier')
ax.set_ylabel('Recall Score')
ax.set_xlabel('Classifiers')


ax.grid(True, linestyle='-', alpha=0.7)
plt.savefig("recall_classs1_SMOTE.pdf", dpi=300, bbox_inches='tight')
# Show the plot
plt.show()


In [ ]:
 recall_scores_class_1

In [ ]:
fig, axes = plt.subplots(len(metrics), len(classifiers), figsize=(20, 15))
for i, metric in enumerate(metrics):
    for j, clf in enumerate(classifiers):
        axes[i, j].boxplot(results[clf][metric])
        axes[i, j].set_title(f'{metric} of {clf}')
        axes[i, j].set_ylabel(metric.capitalize())

# Adding grids to all subplots
for ax in axes.flatten():
    ax.grid(True, linestyle='-', alpha=0.7)

plt.tight_layout()
plt.show()


In [ ]:
X_resampled.shape

In [ ]:
train_df.shape

In [ ]:
5869-3462

In [ ]:
import pandas as pd

# Assuming X_resample and y_resample are already defined
smote_df = pd.concat([X_resampled, y_resampled], axis=1)
smote_df=smote_df[-2407:]
smote_df

In [ ]:
smote_df.to_csv("synthetic_data_smote.csv", index=False)

In [ ]:
table_evaluator = TableEvaluator(train_df, smote_df)

In [ ]:
table_evaluator.visual_evaluation()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Assuming 'train_df' and 'synthetic_data' are your dataframes

# Add a column to distinguish between real and synthetic data
train_df['Type'] = 'Real'
smote_df['Type'] = 'Synthetic'

# Combine the datasets for easier plotting
combined_data = pd.concat([train_df, smote_df])

# List of features to plot (excluding the 'Type' column)
features = combined_data.drop('Type', axis=1).columns

# Create a violin plot for each feature
for feature in features:
    plt.figure(figsize=(10, 6))
    sns.violinplot(x='Type', y=feature, data=combined_data)
    #plt.title(f'{feature}')
    plt.ylabel(feature)
    plt.xlabel('Data Type')
    plt.show()


In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import pandas as pd

# Assuming you have a DataFrame 'combined_data' with a 'Type' column distinguishing real and synthetic data

# Extract features and labels for t-SNE
features = combined_data.drop('Type', axis=1).values
labels = combined_data['Type'].values

# Set the perplexity values you want to test
perplexity_values = [1,5, 30, 50, 100]

# Create subplots
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(15, 15))
axes = axes.flatten()

for i, perplexity in enumerate(perplexity_values):
    # Apply t-SNE with the current perplexity
    tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42)
    transformed_features = tsne.fit_transform(features)

    # Scatter plot
    axes[i].scatter(transformed_features[labels == 'Real', 0],
                    transformed_features[labels == 'Real', 1],
                    c='orange', label='Real Data', alpha=0.5)

    axes[i].scatter(transformed_features[labels == 'Synthetic', 0],
                    transformed_features[labels == 'Synthetic', 1],
                    c='blue', label='Synthetic Data', alpha=0.5)

    axes[i].legend()
    axes[i].set_title(f't-SNE with perplexity={perplexity}')

# Show the plot
plt.tight_layout()
plt.show()


# **Copula**

In [ ]:
pip install sdv

In [ ]:
train_df = pd.concat([X_train, y_train], axis=1)
train_df.shape

In [ ]:
majority_class = train_df[train_df['dialysis'] == 0]  # if it's stored as string
minority_class = train_df[train_df['dialysis'] == 1]

In [ ]:
from sdv.metadata import SingleTableMetadata


from sklearn.metrics import classification_report
from sdv.single_table import GaussianCopulaSynthesizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import warnings
# Assuming metadata is already defined


warnings.filterwarnings("ignore")

# Lists to store results for each classifier
metrics = ['recall', 'precision', 'f1-score', 'accuracy']
classifiers = ['DecisionTree', 'RandomForest', 'SVM', 'LogisticRegression']
results = {clf: {metric: [] for metric in metrics} for clf in classifiers}
# Dictionary to store recall scores of class 1 for each classifier
recall_scores_class_1 = {
    'DecisionTree': [],
    'RandomForest': [],
    'SVM': [],
    'LogisticRegression': []
}

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data=minority_class)
# Run the process 25 times
for _ in range(25):

    synthesizer = GaussianCopulaSynthesizer(metadata)
    synthesizer.fit(minority_class)
    # Generate synthetic data
    synthetic_data_c = synthesizer.sample(num_rows=2407)
    generated_df = pd.DataFrame(data=synthetic_data_c, columns=minority_class.columns)

    # Concatenate synthetic data with original data
    df11 = pd.concat([train_df, generated_df], ignore_index=True)

    # Prepare features and target variable
    X = df11.drop('dialysis', axis=1)
    y = df11['dialysis']


    # Initialize classifiers
    classifiers = {
        'DecisionTree': DecisionTreeClassifier(random_state=42),
        'RandomForest': RandomForestClassifier(random_state=42),
        'SVM': SVC(random_state=42),
        'LogisticRegression': LogisticRegression(random_state=42)
    }

    for name, clf in classifiers.items():
        # Fit classifier
        clf.fit(X, y)

        # Evaluate classifier
        y_pred = clf.predict(X_test)
        report = classification_report(y_test, y_pred, output_dict=True)
        # Store metrics
        results[name]['recall'].append(report['weighted avg']['recall'])
        results[name]['precision'].append(report['weighted avg']['precision'])
        results[name]['f1-score'].append(report['weighted avg']['f1-score'])
        results[name]['accuracy'].append(accuracy_score(y_test, y_pred))
        # Store recall score of class 1
        recall_scores_class_1[name].append(report['1']['recall'])

# Now, recall_scores_class_1 dictionary contains the recall scores of class 1 for all models


In [ ]:
import pandas as pd
import numpy as np
import warnings

from sdv.metadata import SingleTableMetadata
from sdv.single_table import GaussianCopulaSynthesizer

from sklearn.metrics import classification_report, accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")

# Metrics and classifiers
metrics = ['recall', 'precision', 'f1-score', 'accuracy']
classifier_names = ['DecisionTree', 'RandomForest', 'SVM', 'LogisticRegression']

results = {
    clf: {metric: [] for metric in metrics}
    for clf in classifier_names
}

recall_scores_class_1 = {
    'DecisionTree': [],
    'RandomForest': [],
    'SVM': [],
    'LogisticRegression': []
}

# -----------------------------
# Fit Gaussian Copula ONCE
# -----------------------------
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data=minority_class)

synthesizer = GaussianCopulaSynthesizer(metadata)
synthesizer.fit(minority_class)

# Store synthetic datasets for checking
synthetic_runs = []

# -----------------------------
# Generate 25 synthetic samples
# -----------------------------
for run in range(25):

    # Generate synthetic minority-class data
    generated_df = synthesizer.sample(num_rows=2407)

    # Make sure column order matches minority_class
    generated_df = generated_df[minority_class.columns]

    # Since you are generating minority-class samples, force target = 1
    generated_df['dialysis'] = 1

    synthetic_runs.append(generated_df.copy())

    # Concatenate original training data with synthetic minority samples
    df11 = pd.concat([train_df, generated_df], ignore_index=True)

    # Prepare features and target
    X = df11.drop('dialysis', axis=1)
    y = df11['dialysis']

    # Define classifiers
    classifiers = {
        'DecisionTree': DecisionTreeClassifier(random_state=42),
        'RandomForest': RandomForestClassifier(random_state=42),
        'SVM': SVC(random_state=42),
        'LogisticRegression': LogisticRegression(random_state=42, max_iter=1000)
    }

    for name, clf in classifiers.items():

        clf.fit(X, y)

        y_pred = clf.predict(X_test)

        report = classification_report(
            y_test,
            y_pred,
            output_dict=True,
            zero_division=0
        )

        results[name]['recall'].append(report['weighted avg']['recall'])
        results[name]['precision'].append(report['weighted avg']['precision'])
        results[name]['f1-score'].append(report['weighted avg']['f1-score'])
        results[name]['accuracy'].append(accuracy_score(y_test, y_pred))

        # Class 1 recall
        recall_scores_class_1[name].append(report['1']['recall'])

# -----------------------------
# Check whether synthetic datasets are identical
# -----------------------------
print("Checking whether synthetic datasets are identical to run 0:")

for i in range(1, 25):
    print(f"Run 0 vs Run {i}: {synthetic_runs[0].equals(synthetic_runs[i])}")

# -----------------------------
# Print recall values
# -----------------------------
print("\nClass 1 recall scores:")

for name, scores in recall_scores_class_1.items():
    print(name, scores)
    print("Unique recall values:", sorted(set(scores)))

In [ ]:
import matplotlib.pyplot as plt

# Plotting the recall scores for class 1
fig, ax = plt.subplots(figsize=(10, 6))

# Data for plotting
data = [recall_scores_class_1['DecisionTree'],
        recall_scores_class_1['RandomForest'],
        recall_scores_class_1['SVM'],
        recall_scores_class_1['LogisticRegression']]

# Creating the boxplot
ax.boxplot(data)

# Setting the labels for each boxplot
ax.set_xticklabels(['Decision Tree', 'Random Forest', 'SVM', 'Logistic Regression'])
#ax.set_title('Recall Scores of Class 1 for Each Classifier')
ax.set_ylabel('Recall Score')
ax.set_xlabel('Classifiers')


ax.grid(True, linestyle='-', alpha=0.7)
plt.savefig("recall_classs1_copula.pdf", dpi=300, bbox_inches='tight')
# Show the plot
plt.show()


In [ ]:
fig, axes = plt.subplots(len(metrics), len(classifiers), figsize=(20, 15))
for i, metric in enumerate(metrics):
    for j, clf in enumerate(classifiers):
        axes[i, j].boxplot(results[clf][metric])
        axes[i, j].set_title(f'{metric} of {clf}')
        axes[i, j].set_ylabel(metric.capitalize())

# Adding grids to all subplots
for ax in axes.flatten():
    ax.grid(True, linestyle='-', alpha=0.7)

plt.tight_layout()
plt.show()


In [ ]:
recall_scores_class_1

In [ ]:
generated_df.to_csv("synthetic_data_copula.csv", index=False)


In [ ]:
table_evaluator = TableEvaluator(train_df, generated_df)


In [ ]:
table_evaluator.visual_evaluation()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Assuming 'train_df' and 'synthetic_data' are your dataframes

# Add a column to distinguish between real and synthetic data
train_df['Type'] = 'Real'
generated_df['Type'] = 'Synthetic'

# Combine the datasets for easier plotting
combined_data = pd.concat([train_df, generated_df])

# List of features to plot (excluding the 'Type' column)
features = combined_data.drop('Type', axis=1).columns

# Create a violin plot for each feature
for feature in features:
    plt.figure(figsize=(10, 6))
    sns.violinplot(x='Type', y=feature, data=combined_data)
    #plt.title(f'{feature}')
    plt.ylabel(feature)
    plt.xlabel('Data Type')
    plt.show()


In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import pandas as pd

# Assuming you have a DataFrame 'combined_data' with a 'Type' column distinguishing real and synthetic data

# Extract features and labels for t-SNE
features = combined_data.drop('Type', axis=1).values
labels = combined_data['Type'].values

# Set the perplexity values you want to test
perplexity_values = [1,5, 30, 50, 100]

# Create subplots
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(15, 15))
axes = axes.flatten()

for i, perplexity in enumerate(perplexity_values):
    # Apply t-SNE with the current perplexity
    tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42)
    transformed_features = tsne.fit_transform(features)

    # Scatter plot
    axes[i].scatter(transformed_features[labels == 'Real', 0],
                    transformed_features[labels == 'Real', 1],
                    c='orange', label='Real Data', alpha=0.5)

    axes[i].scatter(transformed_features[labels == 'Synthetic', 0],
                    transformed_features[labels == 'Synthetic', 1],
                    c='blue', label='Synthetic Data', alpha=0.5)

    axes[i].legend()
    axes[i].set_title(f't-SNE with perplexity={perplexity}')

# Show the plot
plt.tight_layout()
plt.show()


In [ ]:
pip install urllib3==1.26.7


# **Binary_Copula**

In [ ]:
train_df = pd.concat([X_train, y_train], axis=1)

In [ ]:
train_df.shape

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score, recall_score
import matplotlib.pyplot as plt
import warnings
import pandas as pd
import numpy as np
from scipy.stats import norm

# Simulated dataset (replace this with your real dataset)
np.random.seed(0)

# Step 1: Filter dataset to only have minority class
minority_data = train_df[train_df['dialysis'] == 1].drop(columns=['dialysis'])

# Step 2: Calculate marginal probabilities for each feature for minority class
marginal_probs = minority_data.mean().values

# Step 3: Estimate the correlation matrix for minority class
cor_matrix = minority_data.corr().values
warnings.filterwarnings("ignore")


# Lists to store results for each classifier
metrics = ['recall', 'precision', 'f1-score', 'accuracy']
classifiers = ['DecisionTree', 'RandomForest', 'SVM', 'LogisticRegression']
results = {clf: {metric: [] for metric in metrics} for clf in classifiers}
# Dictionary to store recall scores of class 1 for each classifier
recall_scores_class_1 = {
    'DecisionTree': [],
    'RandomForest': [],
    'SVM': [],
    'LogisticRegression': []
}

def generate_gaussian_copula(cor_matrix, num_samples, marginal_probs):
    num_vars = cor_matrix.shape[0]
    mean = np.zeros(num_vars)
    mvn_samples = np.random.multivariate_normal(mean, cor_matrix, num_samples)
    uniform_samples = norm.cdf(mvn_samples)
    synthetic_data = (uniform_samples < marginal_probs).astype(int)
    return synthetic_data


# Number of synthetic samples to generate
num_synthetic_samples = 2407



# Run the process 25 times
for _ in range(25):
    # Generate synthetic minority samples
    synthetic_minority_samples = generate_gaussian_copula(cor_matrix, num_synthetic_samples, marginal_probs)

    # Convert synthetic samples to DataFrame
    synthetic_minority_df = pd.DataFrame(synthetic_minority_samples, columns=minority_data.columns)

    # Add the dialysis=1 label to synthetic samples
    synthetic_minority_df['dialysis'] = 1

    # Combine the synthetic samples with the original dataset
    balanced_df = pd.concat([train_df, synthetic_minority_df], ignore_index=True)

    # Train Decision Tree
    X = balanced_df.drop('dialysis', axis=1)
    y = balanced_df['dialysis']

    # Initialize classifiers
    classifiers = {
        'DecisionTree': DecisionTreeClassifier(random_state=42),
        'RandomForest': RandomForestClassifier(random_state=42),
        'SVM': SVC(random_state=42),
        'LogisticRegression': LogisticRegression(random_state=42)
    }

    for name, clf in classifiers.items():
        # Fit classifier
        clf.fit(X, y)

        # Evaluate classifier
        y_pred = clf.predict(X_test)
        report = classification_report(y_test, y_pred, output_dict=True)
        # Store metrics
        results[name]['recall'].append(report['weighted avg']['recall'])
        results[name]['precision'].append(report['weighted avg']['precision'])
        results[name]['f1-score'].append(report['weighted avg']['f1-score'])
        results[name]['accuracy'].append(accuracy_score(y_test, y_pred))
        # Store recall score of class 1
        recall_scores_class_1[name].append(report['1']['recall'])

# Now, recall_scores_class_1 dictionary contains the recall scores of class 1 for all models


In [ ]:
import matplotlib.pyplot as plt

# Plotting the recall scores for class 1
fig, ax = plt.subplots(figsize=(10, 6))

# Data for plotting
data = [recall_scores_class_1['DecisionTree'],
        recall_scores_class_1['RandomForest'],
        recall_scores_class_1['SVM'],
        recall_scores_class_1['LogisticRegression']]

# Creating the boxplot
ax.boxplot(data)

# Setting the labels for each boxplot
ax.set_xticklabels(['Decision Tree', 'Random Forest', 'SVM', 'Logistic Regression'])
#ax.set_title('Recall Scores of Class 1 for Each Classifier')
ax.set_ylabel('Recall Score')
ax.set_xlabel('Classifiers')


ax.grid(True, linestyle='-', alpha=0.7)
plt.savefig("recall_classs1_Binary_copula.pdf", dpi=300, bbox_inches='tight')


# Show the plot
plt.show()


In [ ]:
recall_scores_class_1

In [ ]:
fig, axes = plt.subplots(len(metrics), len(classifiers), figsize=(20, 15))
for i, metric in enumerate(metrics):
    for j, clf in enumerate(classifiers):
        axes[i, j].boxplot(results[clf][metric])
        axes[i, j].set_title(f'{metric} of {clf}')
        axes[i, j].set_ylabel(metric.capitalize())

# Adding grids to all subplots
for ax in axes.flatten():
    ax.grid(True, linestyle='-', alpha=0.7)

plt.tight_layout()
plt.show()


In [ ]:
synthetic_minority_df.shape

In [ ]:
synthetic_minority_df.to_csv("synthetic_data_binary_copula.csv", index=False)


In [ ]:
table_evaluator = TableEvaluator(train_df, synthetic_minority_df)


In [ ]:
table_evaluator.visual_evaluation()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Assuming 'train_df' and 'synthetic_data' are your dataframes

# Add a column to distinguish between real and synthetic data
train_df['Type'] = 'Real'
synthetic_minority_df['Type'] = 'Synthetic'

# Combine the datasets for easier plotting
combined_data = pd.concat([train_df, synthetic_minority_df])

# List of features to plot (excluding the 'Type' column)
features = combined_data.drop('Type', axis=1).columns

# Create a violin plot for each feature
for feature in features:
    plt.figure(figsize=(10, 6))
    sns.violinplot(x='Type', y=feature, data=combined_data)
    #plt.title(f'{feature}')
    plt.ylabel(feature)
    plt.xlabel('Data Type')
    plt.show()




In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import pandas as pd

# Assuming you have a DataFrame 'combined_data' with a 'Type' column distinguishing real and synthetic data

# Extract features and labels for t-SNE
features = combined_data.drop('Type', axis=1).values
labels = combined_data['Type'].values

# Set the perplexity values you want to test
perplexity_values = [1,5, 30, 50, 100]

# Create subplots
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(15, 15))
axes = axes.flatten()

for i, perplexity in enumerate(perplexity_values):
    # Apply t-SNE with the current perplexity
    tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42)
    transformed_features = tsne.fit_transform(features)

    # Scatter plot
    axes[i].scatter(transformed_features[labels == 'Real', 0],
                    transformed_features[labels == 'Real', 1],
                    c='orange', label='Real Data', alpha=0.5)

    axes[i].scatter(transformed_features[labels == 'Synthetic', 0],
                    transformed_features[labels == 'Synthetic', 1],
                    c='blue', label='Synthetic Data', alpha=0.5)

    axes[i].legend()
    axes[i].set_title(f't-SNE with perplexity={perplexity}')

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
from google.colab import files
uploaded=files.upload()

In [ ]:
import io
generated_df= pd.read_csv(io.BytesIO(uploaded['synthetic_data_binary_copula.csv']))

In [ ]:
    # Combine the synthetic samples with the original dataset
balanced_df = pd.concat([train_df, generated_df], ignore_index=True)


In [ ]:
balanced_df.head()

In [ ]:
X = balanced_df.drop('dialysis', axis=1)
y = balanced_df['dialysis']

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

clf = DecisionTreeClassifier(max)
clf.fit(X, y)



In [ ]:
plot_tree(clf,
          filled=True,
          rounded=True,
          class_names=['Class 0', 'Class 1'], # Replace with your class names
          feature_names=X.columns) # Replace with your feature names if available
plt.savefig("DT.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

# Assuming your decision tree model is named 'dt_model'

# Plot the decision tree
plt.figure(figsize=(20, 10))  # Adjust the size as needed
plot_tree(clf,
          filled=True,
          rounded=True,
          feature_names=X.columns, # Replace with your actual feature names
          class_names=['Class1', 'Class2'], # Replace with your actual class names
          max_depth=3, # Adjust or remove this for tree depth
          fontsize=10)
plt.savefig("DT.pdf", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
def extract_rules(tree, feature_names):
    left = tree.tree_.children_left
    right = tree.tree_.children_right
    threshold = tree.tree_.threshold
    features = [feature_names[i] for i in tree.tree_.feature]
    value = tree.tree_.value

    def recurse(node, depth, parent_rule):
        indent = "  " * depth
        if threshold[node] != -2:
            print(f"{indent}if {features[node]} <= {threshold[node]:.2f}:")
            recurse(left[node], depth + 1, parent_rule + [(features[node], '<=', threshold[node])])
            print(f"{indent}else:  # if {features[node]} > {threshold[node]:.2f}")
            recurse(right[node], depth + 1, parent_rule + [(features[node], '>', threshold[node])])
        else:
            print(f"{indent}then: {value[node]}")

    recurse(0, 1, [])

# Use this function to extract and print rules
extract_rules(clf, X.columns)


In [ ]:
dt_model=clf

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Styling parameters
plt.style.use('ggplot')  # Using ggplot style for aesthetic enhancements

feature_importances = dt_model.feature_importances_
indices = np.argsort(feature_importances)[::-1]

# Select the top 10 features
top_10_indices = indices[:10]

plt.figure(figsize=(12, 8))  # Slightly larger plot
#plt.title("Top 10 Feature Importances", fontsize=16)
bars = plt.bar(range(len(top_10_indices)), feature_importances[top_10_indices],
               color=plt.cm.viridis(np.linspace(0, 1, 10)),  # Gradient color
               align="center", width=0.5)  # Adjust bar width here

# Add labels on the top of each bar
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval, round(yval, 4), va='bottom')  # Display the importance value on top of each bar

plt.xticks(range(len(top_10_indices)), X.columns[top_10_indices], rotation=45, fontsize=12)
plt.yticks(fontsize=12)
plt.xlim([-1, len(top_10_indices)])
plt.xlabel('Features', fontsize=14)  # X-axis title
plt.ylabel('Importance', fontsize=14)  # Y-axis title
plt.grid(True, which='major', linestyle='--', linewidth=0.5, alpha=0.7)  # Add grid lines for better readability
plt.tight_layout()
# Save the plot as a PDF
plt.savefig("DT_feature.pdf", dpi=300, bbox_inches='tight')

plt.show()


In [ ]:
pip install shap

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Get the feature importances and their corresponding indices
feature_importances = dt_model.feature_importances_
indices = np.argsort(feature_importances)[::-1]

# Select the top 10 features
top_10_indices = indices[:10]
top_10_importances = feature_importances[top_10_indices]
top_10_features = X.columns[top_10_indices]

# Create the heatmap for top 10 features
plt.figure(figsize=(12, 6))  # Adjusted for better display of a single row
sns.heatmap([top_10_importances], annot=True, cmap='coolwarm', cbar=False,
            xticklabels=top_10_features)
#plt.title('Top 10 Feature Importances Heatmap')
plt.yticks([])  # Hide the y-axis labels as they are not informative in this context
plt.xticks(rotation=45)  # Rotate feature names for better readability
plt.savefig("DT_feature.pdf", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
indices = np.argsort(feature_importances)[::-1]

# Select the top 10 features
top_5_indices = indices[:5]

In [ ]:
X.shape

In [ ]:
balanced_df.columns

In [ ]:
balanced_df=balanced_df.drop('Dataset',1)

In [ ]:
from sklearn.decomposition import PCA
import plotly.express as px

# Assuming df is your dataframe with binary features and 'target' is your binary target.
#df = df.drop(['patient_id'], axis=1)

# Apply PCA and keep the first two principal components
pca = PCA(n_components=2)
components = pca.fit_transform(balanced_df.drop('dialysis', axis=1))

# Create a new DataFrame with the principal components
pca_df = pd.DataFrame(components, columns=['PC1', 'PC2'])
pca_df['dialysis'] = balanced_df['dialysis']  # Add the target back in

# Now create a parallel coordinates plot with the principal components
# Use a continuous color scale with only two colors for the binary target
fig = px.parallel_coordinates(
    pca_df,
    color='dialysis',
    dimensions=['PC1', 'PC2'],
    color_continuous_scale=[(0, 'blue'), (1, 'red')]  # Define the two colors for 0 and 1
)

# To ensure that only 0 and 1 are shown in the legend with discrete colors
fig.update_traces(line=dict(colorscale=[(0, 'blue'), (1, 'red')], cmin=0, cmax=1))

fig.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Assuming you have already created the train_df and balanced_df DataFrames

# Calculate class distribution for the original imbalanced dataset
class_distribution_train = train_df['dialysis'].value_counts()

# Calculate class distribution for the balanced dataset
class_distribution_balanced = balanced_df['dialysis'].value_counts()

# Create a donut chart to compare class distribution
fig, ax = plt.subplots(figsize=(8, 6))

labels = ['Not Dialysis', 'Dialysis']
colors = ['#ff9999','#66b3ff']
explode = (0.1, 0)  # Explode the first slice (Not Dialysis)

# Donut chart for the original imbalanced dataset
ax.pie(class_distribution_train, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90, pctdistance=0.85, explode=explode)
circle = plt.Circle((0,0),0.70,fc='white')
fig.gca().add_artist(circle)
ax.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
ax.set_title('Class Distribution (Imbalanced)')

plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Assuming you have already created the balanced_df DataFrame

# Calculate class distribution for the balanced dataset
class_distribution_balanced = balanced_df['dialysis'].value_counts()

# Create a donut chart to visualize the class distribution of the balanced dataset
fig, ax = plt.subplots(figsize=(8, 6))

labels = ['Not Dialysis', 'Dialysis']
colors = ['#ff9999','#66b3ff']
explode = (0.1, 0)  # Explode the first slice (Not Dialysis)

# Donut chart for the balanced dataset
ax.pie(class_distribution_balanced, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90, pctdistance=0.85, explode=explode)
circle = plt.Circle((0,0),0.70,fc='white')
fig.gca().add_artist(circle)
ax.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
ax.set_title('Class Distribution (Balanced)')

plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Assuming you have already created the train_df and balanced_df DataFrames

# Calculate class distribution for the original imbalanced dataset
class_distribution_train = train_df['dialysis'].value_counts()

# Calculate class distribution for the balanced dataset
class_distribution_balanced = balanced_df['dialysis'].value_counts()

# Create a pie chart for the original imbalanced dataset
fig, axs = plt.subplots(1, 2, figsize=(12, 6))
colors = ['#66b3ff', '#ff9999']

axs[0].pie(class_distribution_train, labels=class_distribution_train.index, colors=colors, autopct='%1.1f%%', startangle=90)
axs[0].set_title('Class Distribution (Imbalanced)')

axs[1].pie(class_distribution_balanced, labels=class_distribution_balanced.index, colors=colors, autopct='%1.1f%%', startangle=90)
axs[1].set_title('Class Distribution (Balanced)')

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Preparing the data
data = {
    "RL": [0.45, 0.60, 0.675, 0.725, 0.775],
    "LR": [0.62, 0.65, 0.675, 0.825, 0.825],
    "SVM": [0.57, 0.625, 0.725, 0.675, 0.775],
    "DT": [0.47, 0.575, 0.575, 0.675, 0.775]
}
df = pd.DataFrame(data, index=["Real-Data", "CTGAN", "SMOTE", "Copula", "Binary-Copula"])
df = df.reset_index().melt(id_vars='index', var_name='Model', value_name='Performance')

# Plotting
plt.figure(figsize=(12, 8))
sns.barplot(x='index', y='Performance', hue='Model', data=df)
plt.title('Model Performance Across Different Data Augmentation Techniques')
plt.xlabel('Data Augmentation Technique')
plt.ylabel('Performance Metric')
plt.xticks(rotation=45)
plt.legend(title='Model')
plt.show()


In [ ]:
import plotly.express as px
import pandas as pd

# Preparing the data
data = {
    "RL": [0.45, 0.60, 0.675, 0.725, 0.775],
    "LR": [0.62, 0.65, 0.675, 0.825, 0.825],
    "SVM": [0.57, 0.625, 0.725, 0.675, 0.775],
    "DT": [0.47, 0.575, 0.575, 0.675, 0.775]
}
df = pd.DataFrame(data, index=["Real-Data", "CTGAN", "SMOTE", "Copula", "Binary-Copula"])
df = df.reset_index().melt(id_vars='index', var_name='Model', value_name='Performance')

# Creating an interactive plot
fig = px.bar(df, x='index', y='Performance', color='Model', barmode='group', title="Model Performance Across Data Augmentation Techniques")
fig.update_layout(xaxis_title='Data Augmentation Technique', yaxis_title='Performance Metric')
fig.show()


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import plotly.graph_objects as go
import matplotlib.pyplot as plt
categories = ['Random Forest','Logistic Regression','Support Vecor Machines','Decision Tree']

fig = go.Figure()

fig.add_trace(go.Scatterpolar(
      r=[0.45,0.62,0.57,0.47],
      theta=categories,
      fill='tonext',
      name='Real-data'
))

fig.add_trace(go.Scatterpolar(
      r=[0.60,0.65,0.625,0.575],
      theta=categories,
      fill='tonext',
      name='CTGAN',
      line=dict(color='yellow', width=2)
))

fig.add_trace(go.Scatterpolar(
      r=[0.675,0.675,0.725,0.575],
      theta=categories,
      fill='tonext',
      name='SMOTE'
))
fig.add_trace(go.Scatterpolar(
      r=[0.725,0.825,0.675,0.675],
      theta=categories,
      fill='tonext',
      name='Copula'
))

fig.add_trace(go.Scatterpolar(
      r=[0.775, 0.825, 0.775, 0.775],
      theta=categories,
      fill='tonext',
      name='Vanilla Binary gaussain Copula Synthesis (without LLM)',
      line=dict(color='pink', width=2)
))


fig.add_trace(go.Scatterpolar(
      r=[0.800,0.875,0.85,0.775],
      theta=categories,
      fill='tonext',
      name='Binary gaussain Copula Synthesis (LLM Empowered)',
      line=dict(color='red', width=2)
))
fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 1]
        ),
        angularaxis=dict(
            direction="clockwise",
            rotation=45
        )
    ),
    showlegend=True,
    legend=dict(
        x=0.1,
        y=0.6
    )
)

plt.savefig("recall 1.png", dpi=300, bbox_inches='tight')
fig.show()


In [ ]:
import plotly.graph_objects as go

categories = ['Random Forest', 'Logistic Regression', 'Support Vector Machines', 'Decision Tree']

fig = go.Figure()

fig.add_trace(go.Scatterpolar(
    r=[0.45, 0.62, 0.57, 0.47],
    theta=categories,
    fill='tonext',
    name='Real-data'
))

fig.add_trace(go.Scatterpolar(
    r=[0.60, 0.65, 0.625, 0.575],
    theta=categories,
    fill='tonext',
    name='CTGAN',
    line=dict(color='yellow', width=2)
))

fig.add_trace(go.Scatterpolar(
    r=[0.675, 0.675, 0.725, 0.575],
    theta=categories,
    fill='tonext',
    name='SMOTE'
))

fig.add_trace(go.Scatterpolar(
    r=[0.725, 0.825, 0.675, 0.675],
    theta=categories,
    fill='tonext',
    name='Copula'
))

fig.add_trace(go.Scatterpolar(
    r=[0.775, 0.825, 0.775, 0.775],
    theta=categories,
    fill='tonext',
    name='Vanilla Binary Gaussian Copula Synthesis',
    line=dict(color='red', width=2)
))

fig.add_trace(go.Scatterpolar(
    r=[0.775, 0.825, 0.775, 0.775],
    theta=categories,
    fill='tonext',
    name='Binary Gaussian Copula Synthesis',
    line=dict(color='red', width=2)
))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 1]
        ),
        angularaxis=dict(
            direction="clockwise",
            rotation=45,
            tickfont=dict(
                size=14,
                color='black'
            )
        )
    ),
    showlegend=True,
    legend=dict(
        x=0.1,
        y=0.6
    )
)

# Note: Use fig.write_image to save the figure as an image file instead of plt.savefig
#fig.write_image("recall_1.png", width=800, height=600, scale=2)
fig.show()


In [ ]:
import plotly.graph_objects as go

categories = ['Random Forest', 'Logistic Regression', 'Support Vector Machines', 'Decision Tree']

fig = go.Figure()

fig.add_trace(go.Scatterpolar(
    r=[0.45, 0.62, 0.57, 0.47],
    theta=categories,
    fill='tonext',
    name='Real-data',
    line=dict(color='blue', dash='8', width=2)  # Set dash='solid' for a solid line style
))

fig.add_trace(go.Scatterpolar(
    r=[0.60, 0.65, 0.625, 0.575],
    theta=categories,
    fill='tonext',
    name='CTGAN',
    line=dict(color='yellow', dash='longdashdot', width=2)  # Set dash='solid' for a solid line style
))

fig.add_trace(go.Scatterpolar(
    r=[0.675, 0.675, 0.725, 0.575],
    theta=categories,
    fill='tonext',
    name='SMOTE',
    line=dict(color='orange', dash='dash', width=2)  # Set dash='dash' for a dashed line style
))
fig.add_trace(go.Scatterpolar(
    r=[0.725, 0.825, 0.675, 0.675],
    theta=categories,
    fill='tonext',
    name='Copula',
    line=dict(color='green', dash='dot', width=2)  # Set dash='dot' for a dotted line style
))
fig.add_trace(go.Scatterpolar(
    r=[0.775, 0.825, 0.775, 0.775],
    theta=categories,
    fill='tonext',
    name='Binary-Copula',
    line=dict(color='red', dash='solid', width=2)  # Set dash='dashdot' for a dash-dot line style
))
fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 1]
        ),
        angularaxis=dict(
            direction="clockwise",
            rotation=45
        ),
        bgcolor='lightgray',  # Set the background color of the polar plot
    ),
    paper_bgcolor='white',  # Set the background color of the paper
    showlegend=True,
    legend=dict(
        x=0.229,
        y=1.05
    )
)

fig.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Data
data = {
    "RL": [0.45, 0.60, 0.675, 0.725, 0.775],
    "LR": [0.62, 0.65, 0.675, 0.825, 0.825],
    "SVM": [0.57, 0.625, 0.725, 0.675, 0.775],
    "DT": [0.47, 0.575, 0.575, 0.675, 0.775]
}
index = ["Real-Data", "CTGAN", "SMOTE", "Copula", "Binary-Copula"]

# Create a DataFrame
df = pd.DataFrame(data, index=index)

# Calculate percentage improvement relative to "Real-Data"
baseline = df.loc["Real-Data"]
percentage_improvements = ((df - baseline) / baseline) * 100

# Create a bar plot to visualize percentage improvements
percentage_improvements.plot(kind='bar', figsize=(10, 6), colormap='viridis', rot=0)
plt.title('Percentage Improvement Relative to Real-Data')
plt.xlabel('Methods')
plt.ylabel('Percentage Improvement (%)')
plt.legend(title='Methods', loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Data
data = {
    "RL": [0.45, 0.60, 0.675, 0.725, 0.775],
    "LR": [0.62, 0.65, 0.675, 0.825, 0.825],
    "SVM": [0.57, 0.625, 0.725, 0.675, 0.775],
    "DT": [0.47, 0.575, 0.575, 0.675, 0.775]
}
index = ["Real-Data", "CTGAN", "SMOTE", "Copula", "Binary-Copula"]

# Create a DataFrame
df = pd.DataFrame(data, index=index)

# Calculate improvements relative to "Real-Data"
baseline = df.loc["Real-Data"]
improvements = df.subtract(baseline, axis=1)

# Create a line plot to visualize improvements
improvements.drop("Real-Data").T.plot(kind='line', marker='o', figsize=(10, 6))
plt.title('Improvement Relative to Real-Data')
plt.xlabel('Methods')
plt.ylabel('Improvement')
plt.legend(title='Methods', loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Data
data = {
    "RL": [0.45, 0.60, 0.675, 0.725, 0.775],
    "LR": [0.62, 0.65, 0.675, 0.825, 0.825],
    "SVM": [0.57, 0.625, 0.725, 0.675, 0.775],
    "DT": [0.47, 0.575, 0.575, 0.675, 0.775]
}
index = ["Real-Data", "CTGAN", "SMOTE", "Copula", "Binary-Copula"]

# Create a DataFrame
df = pd.DataFrame(data, index=index)

# Calculate improvements relative to "Real-Data"
baseline = df.loc["Real-Data"]
improvements = df.subtract(baseline, axis=1)

# Create a line chart to visualize improvements
plt.figure(figsize=(10, 6))
for method in improvements.columns:
    plt.plot(improvements.index, improvements[method], marker='o', label=method)

plt.title('Improvement Relative to Real-Data')
plt.xlabel('Methods')
plt.ylabel('Improvement')
plt.legend(title='Methods', loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Data
data = {
    "RF": [0.45, 0.60, 0.675, 0.725,0.77, 0.800],
    "LR": [0.62, 0.65, 0.675, 0.825,0.82, 0.875],
    "SVM": [0.57, 0.625, 0.725, 0.675, 0.77, .850],
    "DT": [0.47, 0.575, 0.575, 0.675,0.77, 0.775]
}
index = ["Real-Data", "CTGAN", "SMOTE", "Copula", "Vanilla Binary Gaussian Copula Synthesis", "Binary Gaussian Copula Synthesis"]

# Create a DataFrame
df = pd.DataFrame(data, index=index)

# Calculate percentage improvement relative to "Real-Data"
baseline = df.loc["Real-Data"]
percentage_improvements = ((df - baseline) / baseline) * 100

# Create a plot to visualize percentage improvements
plt.figure(figsize=(10, 6))
for method in percentage_improvements.columns:
    plt.plot(percentage_improvements.index, percentage_improvements[method], marker='o', label=method)

#plt.title('Percentage Improvement Relative to Real-Data')
plt.xlabel('Methods')
plt.ylabel('Percentage Improvement (%)')
plt.legend(title='Models', loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig("improvement.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Data
data = {
    "RF": [0.45, 0.60, 0.675, 0.725, 0.77, 0.800],
    "LR": [0.62, 0.65, 0.675, 0.825, 0.82, 0.875],
    "SVM": [0.57, 0.625, 0.725, 0.675, 0.77, 0.850],
    "DT": [0.47, 0.575, 0.575, 0.675, 0.77, 0.775]
}
index = ["Real-Data", "CTGAN", "SMOTE", "Copula", "Vanilla BGCS (Without LLM)", "BGCS (LLM Empowered)"]

# Create a DataFrame
df = pd.DataFrame(data, index=index)

# Calculate percentage improvement relative to "Real-Data"
baseline = df.loc["Real-Data"]
percentage_improvements = ((df - baseline) / baseline) * 100

# Create a plot to visualize percentage improvements
plt.figure(figsize=(10, 6))
for method in percentage_improvements.columns:
    plt.plot(percentage_improvements.index, percentage_improvements[method], marker='o', label=method)

plt.xlabel('Methods')
plt.ylabel('Percentage Improvement (%)')
plt.xticks(rotation=45, ha='right')  # Rotate labels
plt.legend(title='Models', loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()  # Adjust layout to fit rotated labels
plt.savefig("improvement.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Data
data = {
    "RL": [0.45, 0.60, 0.675, 0.725, 0.775],
    "LR": [0.62, 0.65, 0.675, 0.825, 0.825],
    "SVM": [0.57, 0.625, 0.725, 0.675, 0.775],
    "DT": [0.47, 0.575, 0.575, 0.675, 0.775]
}
index = ["Real-Data", "CTGAN", "SMOTE", "Copula", "Binary Gaussian Copula Synthesis"]

# Create a DataFrame
df = pd.DataFrame(data, index=index)

# Calculate percentage improvement relative to "Real-Data"
baseline = df.loc["Real-Data"]
percentage_improvements = ((df - baseline) / baseline) * 100

# Create a plot to visualize percentage improvements
plt.figure(figsize=(10, 6))
for method in percentage_improvements.columns:
    plt.plot(percentage_improvements.index, percentage_improvements[method], marker='o', label=method)

# Setting the title of the plot (optional)
# plt.title('Percentage Improvement Relative to Real-Data', fontsize=14)

# Setting the x-axis and y-axis titles with specified font sizes
plt.xlabel('Methods', fontsize=14)  # Adjust the fontsize as needed
plt.ylabel('Percentage Improvement (%)', fontsize=14)  # Adjust the fontsize as needed

plt.legend(title='Models', loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.savefig("improvement.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Data
data = {
    "RF": [0.45, 0.60, 0.675, 0.750, 0.775],
    "LR": [0.62, 0.65, 0.675, 0.775, 0.825],
    "SVM": [0.57, 0.625, 0.725, 0.725, 0.775],
    "DT": [0.47, 0.575, 0.575, 0.725, 0.775]
}

index = [
    "Real-Data",
    "CTGAN",
    "SMOTE",
    "Copula",
    "Binary Gaussian Copula Synthesis"
]

# Create a DataFrame
df = pd.DataFrame(data, index=index)

# Calculate percentage improvement relative to Real-Data
baseline = df.loc["Real-Data"]
percentage_improvements = ((df - baseline) / baseline) * 100

# Create plot
plt.figure(figsize=(10, 6))

for model in percentage_improvements.columns:
    plt.plot(
        percentage_improvements.index,
        percentage_improvements[model],
        marker='o',
        label=model
    )

plt.xlabel('Methods', fontsize=14)
plt.ylabel('Percentage Improvement (%)', fontsize=14)

plt.legend(title='Models', loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.xticks(rotation=20, ha='right')

plt.savefig("improvement.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
pip install -U kaleido

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from math import pi

# Radar chart data
categories = ['Random Forest', 'Logistic Regression', 'Support Vector Machines', 'Decision Tree']
values = [[0.45, 0.62, 0.57, 0.47], [0.60, 0.65, 0.625, 0.575], [0.675, 0.675, 0.725, 0.575],
          [0.725, 0.825, 0.675, 0.675], [0.775, 0.825, 0.775, 0.775]]
labels = ['Real-data', 'CTGAN', 'SMOTE', 'Copula', 'Binary Gaussian Copula Synthesis']
colors = ['blue', 'yellow', 'green', 'purple', 'red']

# Bar chart data
# [Your bar chart data remains the same]

# Set up the subplot grid
fig, axs = plt.subplots(2, 1, figsize=(10, 15))

# Create radar chart
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

ax = plt.subplot(211, polar=True)

# Draw one axe per variable + add labels
plt.xticks(angles[:-1], categories)

# Draw ylabels
ax.set_rlabel_position(0)
plt.yticks([0.25, 0.5, 0.75], ["0.25", "0.5", "0.75"], color="grey", size=7)
plt.ylim(0, 1)

# Plot data and fill with color
for v, label, color in zip(values, labels, colors):
    v += v[:1]
    ax.plot(angles, v, label=label, color=color, linewidth=2)
    ax.fill(angles, v, color=color, alpha=0.3)

ax.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))

# Create bar chart
# [Your bar chart code remains the same]

# Add 'a' and 'b' labels
ax.text(-0.1, 1.1, 'a', transform=ax.transAxes, size=16, weight='bold')
ax2 = axs[1]
ax2.text(-0.1, 1.1, 'b', transform=ax2.transAxes, size=16, weight='bold')

# Show/save plot
plt.tight_layout()
plt.savefig("combined_chart.png")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from math import pi

# Radar chart data
categories = ['Random Forest', 'Logistic Regression', 'Support Vector Machines', 'Decision Tree']
values = [[0.45, 0.62, 0.57, 0.47], [0.60, 0.65, 0.625, 0.575], [0.675, 0.675, 0.725, 0.575],
          [0.725, 0.825, 0.675, 0.675], [0.775, 0.825, 0.775, 0.775]]
labels = ['Real-data', 'CTGAN', 'SMOTE', 'Copula', 'Binary Gaussian Copula Synthesis']

# Bar chart data
data = {
    "RL": [0.45, 0.60, 0.675, 0.725, 0.775],
    "LR": [0.62, 0.65, 0.675, 0.825, 0.825],
    "SVM": [0.57, 0.625, 0.725, 0.675, 0.775],
    "DT": [0.47, 0.575, 0.575, 0.675, 0.775]
}
index = ["Real-Data", "CTGAN", "SMOTE", "Copula", "Binary Gaussian Copula Synthesis"]
df = pd.DataFrame(data, index=index)
baseline = df.loc["Real-Data"]
percentage_improvements = ((df - baseline) / baseline) * 100

# Set up the subplot grid
fig, axs = plt.subplots(2, 1, figsize=(10, 15))

# Create radar chart
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

ax = plt.subplot(211, polar=True)

# Draw one axe per variable + add labels
plt.xticks(angles[:-1], categories, color='grey', size=8)

# Draw ylabels
ax.set_rlabel_position(0)
plt.yticks([0.25, 0.5, 0.75], ["0.25", "0.5", "0.75"], color="grey", size=7)
plt.ylim(0, 1)

# Plot data and fill with color
for v, label in zip(values, labels):
    v += v[:1]
    ax.plot(angles, v, label=label)
    ax.fill(angles, v, alpha=0.3)

ax.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))

# Create bar chart
ax2 = axs[1]
for method in percentage_improvements.columns:
    ax2.plot(percentage_improvements.index, percentage_improvements[method], marker='o', label=method)

ax2.set_xlabel('Methods')
ax2.set_ylabel('Percentage Improvement (%)')
ax2.legend(title='Models', loc='upper left')
ax2.grid(axis='y', linestyle='--', alpha=0.7)

# Add 'a' and 'b' labels
ax.text(-0.1, 1.1, 'a', transform=ax.transAxes, size=16, weight='bold')
ax2.text(-0.1, 1.1, 'b', transform=ax2.transAxes, size=16, weight='bold')

# Show/save plot
plt.tight_layout()
plt.savefig("combined_chart.png")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from math import pi

# Radar chart data
categories = ['Random Forest', 'Logistic Regression', 'Support Vector Machines', 'Decision Tree']
values = [[0.45, 0.62, 0.57, 0.47], [0.60, 0.65, 0.625, 0.575], [0.675, 0.675, 0.725, 0.575],
          [0.725, 0.825, 0.675, 0.675], [0.775, 0.825, 0.775, 0.775]]
labels = ['Real-data', 'CTGAN', 'SMOTE', 'Copula', 'Binary Gaussian Copula Synthesis']

# Bar chart data
data = {
    "RL": [0.45, 0.60, 0.675, 0.725, 0.775],
    "LR": [0.62, 0.65, 0.675, 0.825, 0.825],
    "SVM": [0.57, 0.625, 0.725, 0.675, 0.775],
    "DT": [0.47, 0.575, 0.575, 0.675, 0.775]
}
index = ["Real-Data", "CTGAN", "SMOTE", "Copula", "Binary Gaussian Copula Synthesis"]
df = pd.DataFrame(data, index=index)
baseline = df.loc["Real-Data"]
percentage_improvements = ((df - baseline) / baseline) * 100

# Function to create radar chart
def create_radar_chart(ax, angles, data, color, label):
    ax.plot(angles, data, color=color, linewidth=2, label=label)
    ax.fill(angles, data, color=color, alpha=0.25)

# Number of variables
num_vars = len(categories)

# Compute angle for each category
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()

# The plot is circular, so we need to "complete the loop"
values += [values[0]]
angles += [angles[0]]

# Set up the subplot grid
fig, axs = plt.subplots(2, 1, figsize=(10, 15), subplot_kw=dict(polar=True))

# Radar chart
ax = axs[0]
# Draw one axe per variable and add labels
ax.set_thetagrids(np.degrees(angles), categories)

# Draw data
colors = ['b', 'r', 'g', 'm', 'y']
for val, color, label in zip(values, colors, labels):
    create_radar_chart(ax, angles, val, color, label)

# Add legend
ax.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))

# Bar chart
ax2 = axs[1]
ax2 = plt.subplot(212)
for method in percentage_improvements.columns:
    ax2.plot(percentage_improvements.index, percentage_improvements[method], marker='o', label=method)

ax2.set_xlabel('Methods')
ax2.set_ylabel('Percentage Improvement (%)')
ax2.legend(title='Models', loc='upper left')
ax2.grid(axis='y', linestyle='--', alpha=0.7)

# Add 'a' and 'b' labels
ax.text(-0.1, 1.1, 'a', transform=ax.transAxes, size=16, weight='bold')
ax2.text(-0.1, 1.1, 'b', transform=ax2.transAxes, size=16, weight='bold')

# Show/save plot
plt.tight_layout()
plt.savefig("combined_chart.png")
plt.show()


In [ ]:
recall_scores_class_1_cb=[0.725,
  0.8,
  0.75,
  0.8,
  0.7,
  0.775,
  0.725,
  0.775,
  0.8,
  0.8,
  0.8,
  0.775,
  0.875,
  0.775,
  0.8,
  0.75,
  0.675,
  0.725,
  0.8,
  0.775,
  0.725,
  0.725,
  0.7,
  0.675,
  0.7]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Assuming you have the data in recall_scores_class_1_cb
mean_value = np.mean(recall_scores_class_1_cb)

plt.boxplot(recall_scores_class_1_cb)
#plt.title('Recall Scores of Class 1 across 25 Runs')
plt.ylabel('Recall')

# Plot the reference line
reference = 0.47
plt.axhline(reference, c='r', linestyle='--', label='Real data')

# Plot the mean as a line across the boxplot
plt.axhline(mean_value, c='green', linestyle='--', label=f'Real + Fake')
plt.legend()
plt.savefig("improved.png", dpi=300, bbox_inches='tight')

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Assuming you have the data in recall_scores_class_1_cb
plt.boxplot(recall_scores_class_1_cb, showmeans=True, meanprops=dict(marker='D', markeredgecolor='black', markerfacecolor='firebrick'))
#plt.title('Recall Scores of Class 1 across 25 Runs')
plt.ylabel('Recall')
reference = 0.26
plt.axhline(reference, c='r', linestyle='--', label='Reference (0.26)')
plt.legend()
plt.show()


In [ ]:
plt.boxplot(f1_scores_class_0_cb)
#plt.title('Recall Scores of Class 1 across 25 Runs')
plt.ylabel('Recall')
plt.show()

In [ ]:
plt.boxplot(f1_scores_class_1_cb)
#plt.title('Recall Scores of Class 1 across 25 Runs')
plt.ylabel('Recall')
plt.show()

In [ ]:
plt.boxplot(accuracy_scores_cb)
#plt.title('Recall Scores of Class 1 across 25 Runs')
#plt.ylabel('Recall')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from ctgan import CTGAN

# Helper function for generating Gaussian Copula-based samples
def generate_gaussian_copula(cor_matrix, num_samples, marginal_probs):
    num_vars = cor_matrix.shape[0]
    mean = np.zeros(num_vars)
    mvn_samples = np.random.multivariate_normal(mean, cor_matrix, num_samples)
    uniform_samples = norm.cdf(mvn_samples)
    synthetic_data = (uniform_samples < marginal_probs).astype(int)
    return synthetic_data


# Assume df is your dataset
np.random.seed(0)
minority_data = df[df['dialysis'] == 1].drop(columns=['dialysis'])
cor_matrix = minority_data.corr().values

# Generate copula-based uniform samples
num_copula_samples = 2249
copula_samples = generate_gaussian_copula(cor_matrix, num_copula_samples)

# Transform uniform copula samples back to the original feature space
transformed_copula_samples = norm.cdf(copula_samples)  # This could be inverse CDF, mapping, etc.

# Prepare data for CTGAN
transformed_copula_df = pd.DataFrame(transformed_copula_samples, columns=minority_data.columns)
transformed_copula_df['dialysis'] = 1

# Specify columns to focus on
focused_columns = train_df.columns

# Train CTGAN model on transformed copula samples
ctgan = CTGAN(epochs=38)
ctgan.fit(transformed_copula_df, focused_columns)

# Generate synthetic samples from CTGAN
num_synthetic_samples = 2249
synthetic_data = ctgan.sample(num_synthetic_samples)
synthetic_data['dialysis'] = 1

# Combine with the original dataset to form a balanced dataset
balanced_df = pd.concat([df, synthetic_data], ignore_index=True)
df50=pd.concat([train_df,balanced],axis=1)
X = df50.drop('dialysis', axis=1)  # Replace 'dialysis' with the actual target column name
y = df50['dialysis']
classifier_dt = DecisionTreeClassifier(random_state=42)
classifier_dt.fit(X, y)

# Evaluate Decision Tree
y_pred_dt = classifier_dt.predict(X_test)
accuracyA_dt = accuracy_score(y_test, y_pred_dt)
print("\nDecision Tree:")
print(classification_report(y_test, y_pred_dt))
print(f"Accuracy: {accuracy_dt}")
# Now, `balanced_df` contains the original data along with synthetic minority class samples


In [ ]:
X_test

In [ ]:
from sklearn.metrics import classification_report
import pandas as pd
import numpy as np
from scipy.stats import norm
from ctgan import CTGAN
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

def generate_gaussian_copula(cor_matrix, num_samples, marginal_probs):
    num_vars = cor_matrix.shape[0]
    mean = np.zeros(num_vars)
    mvn_samples = np.random.multivariate_normal(mean, cor_matrix, num_samples)
    uniform_samples = norm.cdf(mvn_samples)
    synthetic_data = (uniform_samples < marginal_probs).astype(int)
    return synthetic_data

np.random.seed(0)
minority_data = df[df['dialysis'] == 1].drop(columns=['dialysis'])
cor_matrix = minority_data.corr().values

# Generate copula-based uniform samples
num_copula_samples = 2249
recall_scores_class_0_cbg = []
recall_scores_class_1_cbg = []
precision_scores_class_0_cbg = []
precision_scores_class_1_cbg= []
f1_scores_class_0_cbg = []
f1_scores_class_1_cbg = []
accuracy_scores_cbg= []

for _ in range(2):
    copula_samples = generate_gaussian_copula(cor_matrix, num_copula_samples,marginal_probs)

# Transform uniform copula samples back to the original feature space
    transformed_copula_samples = norm.cdf(copula_samples)  # This could be inverse CDF, mapping, etc.

# Prepare data for CTGAN
    transformed_copula_df = pd.DataFrame(transformed_copula_samples, columns=minority_data.columns)
    transformed_copula_df['dialysis'] = 1

# Specify columns to focus on
    focused_columns = train_df.columns

# Train CTGAN model on transformed copula samples
    ctgan = CTGAN(epochs=38)
    ctgan.fit(transformed_copula_df, focused_columns)

# Generate synthetic samples from CTGAN
    num_synthetic_samples = 2249
    synthetic_data = ctgan.sample(num_synthetic_samples)
    synthetic_data['dialysis'] = 1

# Combine with the original dataset to form a balanced dataset
    df100= pd.concat([train_df, synthetic_data], ignore_index=True)

    X = df100.drop('dialysis', axis=1)  # Replace 'dialysis' with the actual target column name
    y = df100['dialysis']

    # Train and evaluate Decision Tree
    classifier_dt = DecisionTreeClassifier(random_state=42)
    classifier_dt.fit(X, y)
    y_pred_dt = classifier_dt.predict(X_test)
    report = classification_report(y_test, y_pred_dt, output_dict=True)
    recall_scores_class_0_cbg.append(report['0']['recall'])
    recall_scores_class_1_cbg.append(report['1']['recall'])
    precision_scores_class_0_cbg.append(report['0']['precision'])
    precision_scores_class_1_cbg.append(report['1']['precision'])
    f1_scores_class_0_cbg.append(report['0']['f1-score'])
    f1_scores_class_1_cbg.append(report['1']['f1-score'])
    accuracy_scores_cbg.append(accuracy_score(y_test, y_pred_dt))
    # Optionally print the report or store it
    print("\nDecision Tree, iteration", _ + 1)
    print(classification_report(y_test, y_pred_dt))

# After the loop, you can analyze or average the metrics across all iterations.


In [ ]:
train_df

In [ ]:
df50


In [ ]:
minority_data = train_df[train_df['dialysis'] == 1].drop(columns=['dialysis'])
cor_matrix = minority_data.corr().values

In [ ]:
cor_matrix.shape

In [ ]:
marginal_probs = minority_data.mean().values
marginal_probs.shape

In [ ]:
pip install copulas

In [ ]:
majority_class = train_df[train_df['dialysis'] == 0]  # if it's stored as string
minority_class = train_df[train_df['dialysis'] == 1]

In [ ]:
XX=minority_class.drop('dialysis',1)
yy=minority_class['dialysis']

In [ ]:
XX

In [ ]:
import numpy as np

from copulas.multivariate import GaussianMultivariate

def create_synthetic(X, y):
    """
    This function combines X and y into a single dataset D, models it
    using a Gaussian copula, and generates a synthetic dataset S. It
    returns the new, synthetic versions of X and y.
    """
    dataset = np.concatenate([X, np.expand_dims(y, 1)], axis=1)

    model = GaussianMultivariate()
    model.fit(dataset)

    synthetic = model.sample(2249)

    X = synthetic.values[:, :-1]
    y = synthetic.values[:, -1]

    return X, y

X_synthetic, y_synthetic = create_synthetic(XX, yy)

In [ ]:
y_syn=pd.DataFrame(y_synthetic,columns=['dialysis'])

In [ ]:
y_syn

In [ ]:
x_syn=pd.DataFrame(X_synthetic,columns=X_train.columns)

In [ ]:
balanced = pd.concat([x_syn,y_syn], axis=1)

print(balanced)

In [ ]:
df50=pd.concat([train_df,balanced],axis=1)

In [ ]:
df50

In [ ]:
X = df50.drop('dialysis', axis=1)  # Replace 'dialysis' with the actual target column name
y = df50['dialysis']
classifier_svm = SVC(random_state=42)
classifier_svm.fit(X, y)

# Evaluate SVM
y_pred_svm = classifier_svm.predict(X_test)
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print("\nSupport Vector Machine:")
print(classification_report(y_test, y_pred_svm))
print(f"Accuracy: {accuracy_svm}")

# Create and fit Decision Tree classifier
classifier_dt = DecisionTreeClassifier(random_state=42)
classifier_dt.fit(X, y)

# Evaluate Decision Tree
y_pred_dt = classifier_dt.predict(X_test)
accuracy_dt = accuracy_score(y_test, y_pred_dt)
print("\nDecision Tree:")
print(classification_report(y_test, y_pred_dt))
print(f"Accuracy: {accuracy_dt}")

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm

# Simulated dataset (replace this with your real dataset)
np.random.seed(0)

# Step 1: Filter dataset to only have minority class
minority_data = train_df[train_df['dialysis'] == 1].drop(columns=['dialysis'])

# Step 2: Calculate marginal probabilities for each feature for minority class
marginal_probs = minority_data.mean().values

# Step 3: Estimate the correlation matrix for minority class
cor_matrix = minority_data.corr().values

# Generate synthetic data using Gaussian Copula (Step 4)
def generate_gaussian_copula(cor_matrix, num_samples, marginal_probs):
    num_vars = cor_matrix.shape[0]
    mean = np.zeros(num_vars)
    mvn_samples = np.random.multivariate_normal(mean, cor_matrix, num_samples)
    uniform_samples = norm.cdf(mvn_samples)
    synthetic_data = (uniform_samples < marginal_probs).astype(int)
    return synthetic_data

# Number of synthetic samples to generate
num_synthetic_samples = 2229

# Generate synthetic minority samples
synthetic_minority_samples = generate_gaussian_copula(cor_matrix, num_synthetic_samples, marginal_probs)

# Convert synthetic samples to DataFrame
synthetic_minority_df = pd.DataFrame(synthetic_minority_samples, columns=minority_data.columns)

# Add the dialysis=1 label to synthetic samples
synthetic_minority_df['dialysis'] = 1

# Combine the synthetic samples with the original dataset
balanced_df = pd.concat([train_df, synthetic_minority_df], ignore_index=True)

# Now, `balanced_df` contains the original data along with the synthetic minority class samples


In [ ]:
pip install CTGAN

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from ctgan import CTGAN

# Helper function for generating Gaussian Copula-based samples
def generate_gaussian_copula(cor_matrix, num_samples):
    num_vars = cor_matrix.shape[0]
    mean = np.zeros(num_vars)
    mvn_samples = np.random.multivariate_normal(mean, cor_matrix, num_samples)
    uniform_samples = norm.cdf(mvn_samples)
    return uniform_samples

# Assume df is your dataset
np.random.seed(0)
minority_data = train_df[train_df['dialysis'] == 1].drop(columns=['dialysis'])
cor_matrix = minority_data.corr().values

# Generate copula-based uniform samples
num_copula_samples = 500
copula_samples = generate_gaussian_copula(cor_matrix, num_copula_samples)

# Transform uniform copula samples back to the original feature space
# Here you might use inverse of CDFs, other mapping techniques or even keep it as is
# For simplicity, let's assume we directly use the copula samples
#transformed_copula_samples = norm.cdf(copula_samples)  # This could be inverse CDF, mapping, etc.
transformed_copula_samples = (copula_samples)
# Prepare data for CTGAN
transformed_copula_df = pd.DataFrame(transformed_copula_samples, columns=minority_data.columns)
transformed_copula_df['dialysis'] = 1

# Train CTGAN model on transformed copula samples
ctgan = CTGAN(epochs=20)
ctgan.fit(transformed_copula_df)

# Generate synthetic samples from CTGAN
num_synthetic_samples = 2229
synthetic_data = ctgan.sample(num_synthetic_samples)
synthetic_data['dialysis'] = 1

# Combine with the original dataset to form a balanced dataset
balanced_df = pd.concat([train_df, synthetic_data], ignore_index=True)

# Now, `balanced_df` contains the original data along with synthetic minority class samples


In [ ]:
X = balanced_df.drop('dialysis', axis=1)  # Replace 'dialysis' with the actual target column name
y = balanced_df['dialysis']
classifier_svm = SVC(random_state=42)
classifier_svm.fit(X, y)

# Evaluate SVM
y_pred_svm = classifier_svm.predict(X_test)
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print("\nSupport Vector Machine:")
print(classification_report(y_test, y_pred_svm))
print(f"Accuracy: {accuracy_svm}")

# Create and fit Decision Tree classifier
classifier_dt = DecisionTreeClassifier(random_state=42)
classifier_dt.fit(X, y)

# Evaluate Decision Tree
y_pred_dt = classifier_dt.predict(X_test)
accuracy_dt = accuracy_score(y_test, y_pred_dt)
print("\nDecision Tree:")
print(classification_report(y_test, y_pred_dt))
print(f"Accuracy: {accuracy_dt}")

In [ ]:
balanced_df.shape

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from ctgan import CTGAN

# Helper function for generating Gaussian Copula-based samples
def generate_gaussian_copula(cor_matrix, num_samples):
    num_vars = cor_matrix.shape[0]
    mean = np.zeros(num_vars)
    mvn_samples = np.random.multivariate_normal(mean, cor_matrix, num_samples)
    uniform_samples = norm.cdf(mvn_samples)
    return uniform_samples

# Assume df is your dataset
np.random.seed(0)
minority_data = df[df['dialysis'] == 1].drop(columns=['dialysis'])
cor_matrix = minority_data.corr().values

# Generate copula-based uniform samples
num_copula_samples = 5000
copula_samples = generate_gaussian_copula(cor_matrix, num_copula_samples)

# Transform uniform copula samples back to the original feature space
transformed_copula_samples = norm.cdf(copula_samples)  # This could be inverse CDF, mapping, etc.

# Prepare data for CTGAN
transformed_copula_df = pd.DataFrame(transformed_copula_samples, columns=minority_data.columns)
transformed_copula_df['dialysis'] = 1

# Specify columns to focus on
focused_columns = ['E87.6', 'R06.02','R60.0','E87.5','E87.6']

# Train CTGAN model on transformed copula samples
ctgan = CTGAN(epochs=20)
ctgan.fit(transformed_copula_df, focused_columns)

# Generate synthetic samples from CTGAN
num_synthetic_samples = 2477
synthetic_data = ctgan.sample(num_synthetic_samples)
synthetic_data['dialysis'] = 1

# Combine with the original dataset to form a balanced dataset
balanced_df = pd.concat([df, synthetic_data], ignore_index=True)

# Now, `balanced_df` contains the original data along with synthetic minority class samples


In [ ]:
transformed_copula_df

In [ ]:
balanced_df.shape

In [ ]:
X = balanced_df.drop('dialysis', axis=1)  # Replace 'dialysis' with the actual target column name
y = balanced_df['dialysis']
X_train, X_test4, y_train, y_test4= train_test_split(X, y, test_size=0.10, random_state=42)



#classifier = LogisticRegression()
classifier = RandomForestClassifier(random_state=42)
classifier.fit(X, y)

# Step 8: Evaluate the trained model
y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)


print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy}")

In [ ]:
X = balanced_df.drop('dialysis', axis=1)  # Replace 'dialysis' with the actual target column name
y = balanced_df['dialysis']
X_train, X_test4, y_train, y_test4= train_test_split(X, y, test_size=0.10, random_state=42)



#classifier = LogisticRegression()
classifier = RandomForestClassifier(random_state=42)
classifier.fit(X_train, y_train)

# Step 8: Evaluate the trained model
y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)


print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy}")

In [ ]:
classifier_svm = SVC(random_state=42)
classifier_svm.fit(X, y)

# Evaluate SVM
y_pred_svm = classifier_svm.predict(X_test)
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print("\nSupport Vector Machine:")
print(classification_report(y_test, y_pred_svm))
print(f"Accuracy: {accuracy_svm}")

# Create and fit Decision Tree classifier
classifier_dt = DecisionTreeClassifier(random_state=42)
classifier_dt.fit(X, y)

# Evaluate Decision Tree
y_pred_dt = classifier_dt.predict(X_test)
accuracy_dt = accuracy_score(y_test, y_pred_dt)
print("\nDecision Tree:")
print(classification_report(y_test, y_pred_dt))
print(f"Accuracy: {accuracy_dt}")

In [ ]:
classifier_svm = SVC(random_state=42)
classifier_svm.fit(X_train, y_train)

# Evaluate SVM
y_pred_svm = classifier_svm.predict(X_test)
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print("\nSupport Vector Machine:")
print(classification_report(y_test, y_pred_svm))
print(f"Accuracy: {accuracy_svm}")

# Create and fit Decision Tree classifier
classifier_dt = DecisionTreeClassifier(random_state=42)
classifier_dt.fit(X_train, y_train)

# Evaluate Decision Tree
y_pred_dt = classifier_dt.predict(X_test)
accuracy_dt = accuracy_score(y_test, y_pred_dt)
print("\nDecision Tree:")
print(classification_report(y_test, y_pred_dt))
print(f"Accuracy: {accuracy_dt}")

In [ ]:
balanced_df.shape

In [ ]:
from sklearn.svm import SVC  # Support Vector Machine
from sklearn.tree import DecisionTreeClassifier  # Decision Tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
pip install tensorflow

In [ ]:
classifier_svm = SVC(random_state=42)
classifier_svm.fit(X_train, y_train)

# Evaluate SVM
y_pred_svm = classifier_svm.predict(X_test)
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print("\nSupport Vector Machine:")
print(classification_report(y_test, y_pred_svm))
print(f"Accuracy: {accuracy_svm}")

# Create and fit Decision Tree classifier
classifier_dt = DecisionTreeClassifier(random_state=42)
classifier_dt.fit(X_train, y_train)

# Evaluate Decision Tree
y_pred_dt = classifier_dt.predict(X_test)
accuracy_dt = accuracy_score(y_test, y_pred_dt)
print("\nDecision Tree:")
print(classification_report(y_test, y_pred_dt))
print(f"Accuracy: {accuracy_dt}")

In [ ]:
import pandas as pd
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression


In [ ]:
# Split the data into X (features) and y (target)
X = df.drop('dialysis', axis=1)
y = df['dialysis']


In [ ]:
# Initialize a logistic regression classifier
estimator = LogisticRegression(solver='liblinear')

# Initialize RFE and specify the number of features to select
num_features_to_select = 60
selector = RFE(estimator, n_features_to_select=num_features_to_select)

# Fit RFE
selector = selector.fit(X, y)



In [ ]:
# Get the boolean mask of selected features
selected_mask = selector.support_

# Get the features selected by RFE
selected_features = X.columns[selected_mask]
print("Selected features:", selected_features)


In [ ]:
# Use only the selected features for X
X_selected = X[selected_features]

# Split the data into training and testing sets
X_train, X_test11, y_train, y_test11 = train_test_split(X_selected, y, test_size=0.2, random_state=42)




classifier = LogisticRegression()
classifier.fit(X_train, y_train)

# Step 8: Evaluate the trained model
y_pred = classifier.predict(X_test[selected_features])
accuracy = accuracy_score(y_test, y_pred)


print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy}")

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
ConfusionMatrixDisplay.from_estimator(
    classifier, X_test, y_test, xticks_rotation="vertical"
)
plt.tight_layout()
plt.show()

In [ ]:
X = train_df.drop('dialysis', axis=1)
y = train_df['dialysis']


# **Smote**


In [ ]:
from imblearn.over_sampling import SMOTE
from collections import Counter
# Define the resampling method
resample = SMOTE(sampling_strategy=0.8)
X = train_df.drop('dialysis', axis=1)
y = train_df['dialysis']
# Fit the resampling method to the data
X, y = resample.fit_resample(X, y)

# Count the new class distribution
counter = Counter(y)
print(counter)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
# For df1
percentage_X82248_df1 = df['X82248'].mean() * 100
percentage_X82565_df1 = df['X82565'].mean() * 100
percentage_both_df1 = df[(df['X82248'] == 1) & (df['X82565'] == 1)].shape[0] / df.shape[0] * 100

# For df2
percentage_X82248_df2 = X['X82248'].mean() * 100
percentage_X82565_df2 = X['X82565'].mean() * 100
percentage_both_df2 = X[(X['X82248'] == 1) & (X['X82565'] == 1)].shape[0] / X.shape[0] * 100

# Data to plot
labels = ['X82248', 'X82565', 'Both']
percentages_df1 = [percentage_X82248_df1, percentage_X82565_df1, percentage_both_df1]
percentages_df2 = [percentage_X82248_df2, percentage_X82565_df2, percentage_both_df2]

# X-axis locations for groups
x = np.arange(len(labels))

# Width of the bars
width = 0.35

# Create grouped bar plot
fig, ax = plt.subplots()
rects1 = ax.bar(x - width/2, percentages_df1, width, label='Real Data')
rects2 = ax.bar(x + width/2, percentages_df2, width, label='Final Data')

# Adding title and labels
ax.set_title('Distribution of Features for Two Datasets')
ax.set_xlabel('Features')
ax.set_ylabel('Percentage (%)')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

# Show the plot
plt.show()


In [ ]:
X.head()

In [ ]:
X_train, X_test1, y_train, y_test1 = train_test_split(X, y, test_size=0.01, random_state=42)



#classifier = LogisticRegression()
classifier = RandomForestClassifier(random_state=42)
classifier.fit(X, y)

# Step 8: Evaluate the trained model
y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)


print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy}")

In [ ]:
classifier_svm = SVC(random_state=42)
classifier_svm.fit(X, y)

# Evaluate SVM
y_pred_svm = classifier_svm.predict(X_test)
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print("\nSupport Vector Machine:")
print(classification_report(y_test, y_pred_svm))
print(f"Accuracy: {accuracy_svm}")

# Create and fit Decision Tree classifier
classifier_dt = DecisionTreeClassifier(random_state=42)
classifier_dt.fit(X, y)

# Evaluate Decision Tree
y_pred_dt = classifier_dt.predict(X_test)
accuracy_dt = accuracy_score(y_test, y_pred_dt)
print("\nDecision Tree:")
print(classification_report(y_test, y_pred_dt))
print(f"Accuracy: {accuracy_dt}")

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
ConfusionMatrixDisplay.from_estimator(
    classifier, X_test, y_test, xticks_rotation="vertical"
)
plt.tight_layout()
plt.show()

In [ ]:
pip install CTGAN

In [ ]:
from ctgan import CTGAN
from ctgan import load_demo

df20=df[df['dialysis']==1]
# Names of the columns that are discrete
#discrete_columns = df.columns
discrete_columns = df[['E87.6', 'R06.02','R60.0','E87.5','E87.6']]
ctgan = CTGAN(epochs=20)
ctgan.fit(df20, discrete_columns)

# Create synthetic data
synthetic_data = ctgan.sample(2477)
import pandas as pd

# Assuming you have two DataFrames df1 and df2 with the same columns
# Here's how you can merge them

# Concatenate the DataFrames vertically (along rows) to create a new DataFrame with both datasets combined
df30 = pd.concat([df,synthetic_data ], ignore_index=True)

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from ctgan import CTGAN

# Assume df is your original dataset
# Simulated dataset (replace this with your real dataset)
np.random.seed(0)

# Step 1: Generate synthetic minority samples using Gaussian Copula
def generate_gaussian_copula(cor_matrix, num_samples, marginal_probs):
    num_vars = cor_matrix.shape[0]
    mean = np.zeros(num_vars)
    mvn_samples = np.random.multivariate_normal(mean, cor_matrix, num_samples)
    uniform_samples = norm.cdf(mvn_samples)
    synthetic_data = (uniform_samples < marginal_probs).astype(int)
    return synthetic_data

minority_data = df[df['dialysis'] == 1].drop(columns=['dialysis'])
marginal_probs = minority_data.mean().values
cor_matrix = minority_data.corr().values
num_synthetic_samples = 1238

synthetic_minority_samples_copula = generate_gaussian_copula(cor_matrix, num_synthetic_samples, marginal_probs)
synthetic_minority_df_copula = pd.DataFrame(synthetic_minority_samples_copula, columns=minority_data.columns)
synthetic_minority_df_copula['dialysis'] = 1

# Step 2: Generate synthetic minority samples using CTGAN
df20 = df[df['dialysis'] == 1]
# Uncomment the below line if you know the discrete columns, otherwise CTGAN will consider all as continuous
# discrete_columns = ['E87.6', 'R06.02', 'R60.0', 'E87.5', 'E87.6']

ctgan = CTGAN(epochs=20)
ctgan.fit(df20)  # , discrete_columns)  # Uncomment and add discrete_columns if needed

synthetic_data_ctgan = ctgan.sample(num_synthetic_samples)
synthetic_data_ctgan['dialysis'] = 1

# Step 3: Combine synthetic samples with the original dataset
balanced_df = pd.concat([df, synthetic_minority_df_copula, synthetic_data_ctgan], ignore_index=True)

# Now, `balanced_df` contains the original data along with synthetic minority class samples generated by both Gaussian Copula and CTGAN


In [ ]:
X = df30.drop('dialysis', axis=1)  # Replace 'dialysis' with the actual target column name
y = df30['dialysis']

X_train, X_test1, y_train, y_test1 = train_test_split(X, y, test_size=0.1, random_state=42)



#classifier = LogisticRegression()
classifier = RandomForestClassifier(random_state=42)
classifier.fit(X_train, y_train)

# Step 8: Evaluate the trained model
y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)


print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy}")

In [ ]:
classifier_svm = SVC(random_state=42)
classifier_svm.fit(X_train, y_train)

# Evaluate SVM
y_pred_svm = classifier_svm.predict(X_test)
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print("\nSupport Vector Machine:")
print(classification_report(y_test, y_pred_svm))
print(f"Accuracy: {accuracy_svm}")

# Create and fit Decision Tree classifier
classifier_dt = DecisionTreeClassifier(random_state=42)
classifier_dt.fit(X_train, y_train)

# Evaluate Decision Tree
y_pred_dt = classifier_dt.predict(X_test)
accuracy_dt = accuracy_score(y_test, y_pred_dt)
print("\nDecision Tree:")
print(classification_report(y_test, y_pred_dt))
print(f"Accuracy: {accuracy_dt}")

In [ ]:
df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load your data
#ata = pd.read_csv('your_dataset.csv')  # Change the path to your dataset
data=df
# Separate the data by class
majority_class = data[data['dialysis'] == 0]  # if it's stored as string
minority_class = data[data['dialysis'] == 1]  # if it's stored as string



def analyze_distribution(feature_name):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))

    # Define a common order for both plots
    order = [0, 1] if data[feature_name].dtype == 'int64' else ['0', '1']

    sns.countplot(x=feature_name, data=majority_class, ax=ax[0], order=order)
    ax[0].set_title(f"Distribution of {feature_name} in Majority Class")

    # If a category is missing in the minority class data, append it as dummy data
    if minority_class[feature_name].nunique() < 2:
        dummy_data = pd.DataFrame({feature_name: order}, dtype=str)
        plot_data = pd.concat([minority_class, dummy_data])
    else:
        plot_data = minority_class

    sns.countplot(x=feature_name, data=plot_data, ax=ax[1], order=order)
    ax[1].set_title(f"Distribution of {feature_name} in Minority Class")

    plt.tight_layout()
    plt.show()

for feature in data.columns:
    if feature != 'dialysis':  # We don't want to include the target in this analysis
        analyze_distribution(feature)


    plt.tight_layout()
    plt.show()


for feature in data.columns:
    if feature != 'dialysis':  # We don't want to include the target in this analysis
        analyze_distribution(feature)

# To visualize relationships between features, you can use pairplots or correlation matrices.
# Given that the dataset is binary, a correlation matrix might be more insightful.

correlation_matrix = data.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.title("Feature Correlation Matrix")
plt.show()


In [ ]:
#E87.6, J90, R06.00, R06.02, R60.0,E87.5,E87.6
X87040,
 Separate the data by class
majority_class = data[data['dialysis'] == 0]  # if it's stored as string
minority_class = data[data['dialysis'] == 1]
minority_class

In [ ]:
model = CTGAN(epochs=10)
model.fit(df20, discrete_columns)

In [ ]:
conditions = pd.DataFrame({
    'E87.6': [1,1,1],
    })

model.sample_remaining_columns(conditions)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
ConfusionMatrixDisplay.from_estimator(
    classifier, X_test, y_test, xticks_rotation="vertical"
)
plt.tight_layout()
plt.show()

In [ ]:
from imblearn.over_sampling import SMOTE
from collections import Counter
# Define the resampling method
resample = SMOTE(sampling_strategy=0.7)

# Fit the resampling method to the data
X, y = resample.fit_resample(X, y)

# Count the new class distribution
counter = Counter(y)
print(counter)


In [ ]:
E87.6, J90, R06.00, R06.02, R60.0,E87.5,E87.6

In [ ]:
df['E87.6'].mean()

In [ ]:
df['J90'].mean()

In [ ]:
df['R06.00'].mean()

In [ ]:
minority_class = data[data['dialysis'] == 1]

In [ ]:
minority_class['R06.00'].mean()

In [ ]:
minority_class['J90'].mean()

In [ ]:
minority_class['E87.6'].mean()

In [ ]:
minority_class['R06.02'].mean()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LeakyReLU
from tensorflow.keras.optimizers import Adam

# Your data preparation step here
# I am assuming that you have a dataframe df with 'dialysis' as the target variable
X = train_df.drop('dialysis', axis=1)
y = train_df['dialysis']

majority_class = train_df[y == 0]
minority_class = train_df[y == 1]

# GAN components
def define_generator(latent_dim, n_outputs):
    model = Sequential()
    model.add(Dense(10, input_dim=latent_dim))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(n_outputs, activation='sigmoid'))
    return model

def define_discriminator(n_inputs):
    model = Sequential()
    model.add(Dense(25, input_dim=n_inputs))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer=Adam(lr=0.001), metrics=['accuracy'])
    return model

def define_gan(generator, discriminator):
    discriminator.trainable = False
    model = Sequential()
    model.add(generator)
    model.add(discriminator)
    model.compile(loss='binary_crossentropy', optimizer=Adam(lr=0.001))
    return model

# data for GAN
def generate_real_samples(dataset, n_samples):
    ix = np.random.randint(0, dataset.shape[0], n_samples)
    X = dataset.iloc[ix]
    y = np.ones((n_samples, 1))
    return X, y

def generate_latent_points(latent_dim, n_samples):
    x_input = np.random.randn(latent_dim * n_samples)
    x_input = x_input.reshape(n_samples, latent_dim)
    return x_input

def generate_fake_samples(generator, latent_dim, n_samples):
    x_input = generate_latent_points(latent_dim, n_samples)
    X = generator.predict(x_input)
    y = np.zeros((n_samples, 1))
    return X, y

# GAN training
def train_gan(generator, discriminator, gan_model, latent_dim, n_epochs, n_batch, majority_data):
    half_batch = int(n_batch / 2)
    g_losses = []
    d_losses = []
    real_ratio = 0.5
    fake_ratio = 1 - real_ratio
    for epoch in range(n_epochs):
        for _ in range(2):
            # use int() to ensure the result is an integer
            X_real, y_real = generate_real_samples(majority_data, int(half_batch * real_ratio))
            X_fake, y_fake = generate_fake_samples(generator, latent_dim, int(half_batch * fake_ratio))
            d_loss_real, _ = discriminator.train_on_batch(X_real, y_real)
            d_loss_fake, _ = discriminator.train_on_batch(X_fake, y_fake)
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

        X_gan = generate_latent_points(latent_dim, n_batch)
        y_gan = np.ones((n_batch, 1))
        g_loss = gan_model.train_on_batch(X_gan, y_gan)
        print(f'Epoch: {epoch+1}, Discriminator Loss: {d_loss}, Generator Loss: {g_loss}')
        g_losses.append(g_loss)
        d_losses.append(d_loss)

    plt.plot(g_losses, label='Generator Loss')
    plt.plot(d_losses, label='Discriminator Loss')
    plt.title('GAN Training Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()


# Set hyperparameters
latent_dim = 5
n_epochs = 400
n_batch = 63

discriminator = define_discriminator(majority_class.shape[1])
generator = define_generator(latent_dim, majority_class.shape[1])
gan_model = define_gan(generator, discriminator)

# Train the GAN
train_gan(generator, discriminator, gan_model, latent_dim, n_epochs, n_batch, majority_class)

# Generate new minority samples
n_samples = 2249
latent_points = generate_latent_points(latent_dim, n_samples)
synthetic_samples = generator.predict(latent_points)


#feature_names = ['X82248', 'X82565']

#desired_percentages = [0.26, 0.27]

# Adjust the generated data to meet the desired conditions
#for i, feature_name in enumerate(feature_names):
 #   column_index = X.columns.get_loc(feature_name)
  #  column = synthetic_samples[:, column_index]

    # Calculate the desired number of 1s
   # desired_ones = int(n_samples * desired_percentages[i])

    # Set the desired percentage of 1s for the feature
    #synthetic_samples[:, column_index] = np.where(np.arange(n_samples) < desired_ones, 1, 0)

# Adjust the condition of 40% of records having 1 for both features
#num_additional_ones = int(n_samples * 0.14)

#column82248 = synthetic_samples[:, X.columns.get_loc('X82248')]
#column82565 = synthetic_samples[:, X.columns.get_loc('X82565')]

# Set 40% of records to have 1 for both features
#combined_indices = np.logical_and(column82248 == 1, column82565 == 1)
#combined_indices = np.random.choice(np.where(combined_indices)[0], size=num_additional_ones, replace=False)
#synthetic_samples[combined_indices, X.columns.get_loc('X82248')] = 1
#synthetic_samples[combined_indices, X.columns.get_loc('X82565')] = 1


# Generated data (synthetic samples)
generated_data = synthetic_samples.copy()
# Combine with existing minority samples
combined_samples = np.concatenate((minority_class, synthetic_samples))

# Combine with majority samples for balanced dataset
n_majority_samples = combined_samples.shape[0]
balanced_X = np.concatenate((combined_samples, majority_class.sample(n_majority_samples)))
balanced_y = np.concatenate((np.ones(n_majority_samples), np.zeros(n_majority_samples)))




In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LeakyReLU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Add, Multiply, Attention
# Your data preparation step here
# I am assuming that you have a dataframe df with 'dialysis' as the target variable
X = train_df.drop('dialysis', axis=1)
y = train_df['dialysis']

majority_class = train_df[y == 0]
minority_class = train_df[y == 1]

# GAN components
from tensorflow.keras.layers import BatchNormalization, Dropout

# GAN Components
from tensorflow.keras.layers import Add, Multiply, Attention

# For Generator
import numpy as np
from tensorflow.keras.layers import BatchNormalization, Dropout

# Your data preparation step here
# I am assuming that you have a dataframe df with 'dialysis' as the target variable
# X = df.drop('dialysis', axis=1)
# y = df['dialysis']

# Majority and Minority classes
# majority_class = df[y == 0]
# minority_class = df[y == 1]

# Modified Generator
def define_generator(latent_dim, n_outputs):
    model = Sequential()
    model.add(Dense(50, input_dim=latent_dim))
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization())

    model.add(Dense(50))
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization())

    model.add(Dense(n_outputs, activation='sigmoid'))
    return model

# Modified Discriminator
def define_discriminator(n_inputs):
    model = Sequential()
    model.add(Dense(512, input_dim=n_inputs))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dropout(0.4))
    model.add(Dense(256))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dropout(0.4))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer=Adam(lr=0.0002, beta_1=0.5), metrics=['accuracy'])
    return model


def define_gan(generator, discriminator):
    discriminator.trainable = False
    model = Sequential()
    model.add(generator)
    model.add(discriminator)
    model.compile(loss='binary_crossentropy', optimizer=Adam(lr=0.001))
    return model

# data for GAN
def generate_real_samples(dataset, n_samples):
    ix = np.random.randint(0, dataset.shape[0], n_samples)
    X = dataset.iloc[ix]
    y = np.ones((n_samples, 1))
    return X, y

def generate_latent_points(latent_dim, n_samples):
    x_input = np.random.randn(latent_dim * n_samples)
    x_input = x_input.reshape(n_samples, latent_dim)
    return x_input

def generate_fake_samples(generator, latent_dim, n_samples):
    x_input = generate_latent_points(latent_dim, n_samples)
    X = generator.predict(x_input)
    y = np.zeros((n_samples, 1))
    return X, y

# GAN training
def train_gan(generator, discriminator, gan_model, latent_dim, n_epochs, n_batch, majority_data):
    half_batch = int(n_batch / 2)
    g_losses = []
    d_losses = []
    real_ratio = 0.5
    fake_ratio = 1 - real_ratio
    for epoch in range(n_epochs):
        for _ in range(2):
            # use int() to ensure the result is an integer
            X_real, y_real = generate_real_samples(majority_data, int(half_batch * real_ratio))
            X_fake, y_fake = generate_fake_samples(generator, latent_dim, int(half_batch * fake_ratio))
            d_loss_real, _ = discriminator.train_on_batch(X_real, y_real)
            d_loss_fake, _ = discriminator.train_on_batch(X_fake, y_fake)
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

        X_gan = generate_latent_points(latent_dim, n_batch)
        y_gan = np.ones((n_batch, 1))
        g_loss = gan_model.train_on_batch(X_gan, y_gan)
        print(f'Epoch: {epoch+1}, Discriminator Loss: {d_loss}, Generator Loss: {g_loss}')
        g_losses.append(g_loss)
        d_losses.append(d_loss)

    plt.plot(g_losses, label='Generator Loss')
    plt.plot(d_losses, label='Discriminator Loss')
    plt.title('GAN Training Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()


# Set hyperparameters
latent_dim = 50
n_epochs = 800
n_batch = 63

discriminator = define_discriminator(majority_class.shape[1])
generator = define_generator(latent_dim, majority_class.shape[1])
gan_model = define_gan(generator, discriminator)

# Train the GAN
train_gan(generator, discriminator, gan_model, latent_dim, n_epochs, n_batch, majority_class)

# Generate new minority samples
n_samples = 2249
latent_points = generate_latent_points(latent_dim, n_samples)
synthetic_samples = generator.predict(latent_points)

#feature_names = ['J90','R06.02']
#feature_names = ['J90','E87.6']
#desired_percentages = [0.62,0.71]
#desired_percentages = [0.62,0.66]
# Adjust the generated data to meet the desired conditions
#for i, feature_name in enumerate(feature_names):
 # column_index = X.columns.get_loc(feature_name)
 # column = synthetic_samples[:, column_index]

    # Calculate the desired number of 1s
 # desired_ones = int(n_samples * desired_percentages[i])

    # Set the desired percentage of 1s for the feature
#synthetic_samples[:, column_index] = np.where(np.arange(n_samples) < desired_ones, 1, 0)

# Adjust the condition of 40% of records having 1 for both features
#num_additional_ones = int(n_samples * 0.4)

#column82248 = synthetic_samples[:, X.columns.get_loc('J90')]
#column82565 = synthetic_samples[:, X.columns.get_loc('R06.00')]
#column3 = synthetic_samples[:, X.columns.get_loc('E87.6')]
# Set 40% of records to have 1 for both features
#combined_indices = np.logical_and(column82248 == 1, column82565 == 1)
#combined_indices = np.random.choice(np.where(combined_indices)[0], size=num_additional_ones, replace=False)
#synthetic_samples[combined_indices, X.columns.get_loc('X82248')] = 1
#synthetic_samples[combined_indices, X.columns.get_loc('X82565')] = 1

# Generated data (synthetic samples)
generated_data = synthetic_samples.copy()
# Combine with existing minority samples
combined_samples = np.concatenate((minority_class, synthetic_samples))

# Combine with majority samples for balanced dataset
n_majority_samples = combined_samples.shape[0]
balanced_X = np.concatenate((combined_samples, majority_class.sample(n_majority_samples)))
balanced_y = np.concatenate((np.ones(n_majority_samples), np.zeros(n_majority_samples)))




In [ ]:
from sdv.tabular import CopulaGAN

In [ ]:
pip install sdv

In [ ]:
import sdv
print(sdv.__version__)

In [ ]:
metadata = synthesizer.get_metadata()

In [ ]:
from sdv.metadata import Metadata
metadata = synthesizer.get_metadata()
metadata = Metadata()
metadata = metadata.from_data(train_df)


In [ ]:
from sdv.single_table import GaussianCopulaSynthesizer

# Step 1: Create the synthesizer
synthesizer = GaussianCopulaSynthesizer()

# Step 2: Train the synthesizer
synthesizer.fit(train_df)

# Step 3: Generate synthetic data
synthetic_data = synthesizer.sample(num_rows=100)


from sdv.single_table import CTGANSynthesizer

synthesizer = CTGANSynthesizer(metadata)
synthesizer.fit(minority_class)

synthetic_data = synthesizer.sample(num_rows=10)

In [ ]:
from sdv.metadata import SingleTableMetadata

metadata = SingleTableMetadata()

In [ ]:
from sdv.metadata import Table

table_metadata = Table.from_dataframe(data)


In [ ]:
X=train.df

In [ ]:
from sdv.datasets.local import load_csvs


In [ ]:
from sdv.metadata import SingleTableMetadata

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data=minority_class)

In [ ]:
metadata.detect_from_dataframe(data=minority_class)

In [ ]:
X = train_df.drop('dialysis', axis=1)
y = train_df['dialysis']

majority_class = train_df[y == 0]
minority_class = train_df[y == 1]

In [ ]:
from sdv.single_table import CTGANSynthesizer

synthesizer = CTGANSynthesizer(metadata)
synthesizer.fit(minority_class)

synthetic_data = synthesizer.sample(num_rows=10)

In [ ]:
synthetic_data = synthesizer.sample(num_rows=2249)

In [ ]:
from sdv.single_table import GaussianCopulaSynthesizer

# Step 1: Create the synthesizer
synthesizer = GaussianCopulaSynthesizer(metadata)

# Step 2: Train the synthesizer
synthesizer.fit(minority_class)

# Step 3: Generate synthetic data
synthetic_data = synthesizer.sample(num_rows=2249)

In [ ]:
generated_df = pd.DataFrame(data=synthetic_data, columns=minority_class.columns)

In [ ]:
synthetic_data

In [ ]:
from sdv.single_table import CopulaGANSynthesizer

synthesizer = CopulaGANSynthesizer(metadata)
synthesizer.fit(data)

synthetic_data = synthesizer.sample(num_rows=10)

In [ ]:
generated_data.shape

In [ ]:
generated_df = pd.DataFrame(data=generated_data, columns=minority_class.columns)

In [ ]:
generated_df

In [ ]:
# Calculate the mean of each column in the generated DataFrame
column_means = generated_df.mean()

# Iterate over each column in the generated DataFrame and set values greater than the mean to 1 and others to 0
for column in generated_df.columns:
    generated_df[column] = (generated_df[column] > column_means[column]).astype(int)




In [ ]:
generated_df['dialysis'] = 1
generated_df

In [ ]:
import pandas as pd

# Assuming you have two DataFrames df1 and df2 with the same columns
# Here's how you can merge them

# Concatenate the DataFrames vertically (along rows) to create a new DataFrame with both datasets combined
df10 = pd.concat([train_df, generated_df], ignore_index=True)



In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score


In [ ]:
X = df10.drop('dialysis', axis=1)  # Replace 'dialysis' with the actual target column name
y = df10['dialysis']
X_train, X_test2, y_train, y_test2 = train_test_split(X, y, test_size=0.20, random_state=42)


classifier = LogisticRegression()
classifier.fit(X, y)

# Step 8: Evaluate the trained model
y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy}")

In [ ]:
from sklearn.svm import SVC  # Support Vector Machine
from sklearn.tree import DecisionTreeClassifier  # Decision Tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score


In [ ]:
X = df10.drop('dialysis', axis=1)  # Replace 'dialysis' with the actual target column name
y = df10['dialysis']
X_train, X_test5, y_train, y_test5 = train_test_split(X, y, test_size=0.20, random_state=42)

# Create a Random Forest classifier instance
classifier = RandomForestClassifier(random_state=42)

# Fit the model
classifier.fit(X, y)

# Evaluate the trained model
y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy}")


In [ ]:
classifier_svm = SVC(random_state=42)
classifier_svm.fit(X, y)

# Evaluate SVM
y_pred_svm = classifier_svm.predict(X_test)
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print("\nSupport Vector Machine:")
print(classification_report(y_test, y_pred_svm))
print(f"Accuracy: {accuracy_svm}")

# Create and fit Decision Tree classifier
classifier_dt = DecisionTreeClassifier(random_state=42)
classifier_dt.fit(X, y)

# Evaluate Decision Tree
y_pred_dt = classifier_dt.predict(X_test)
accuracy_dt = accuracy_score(y_test, y_pred_dt)
print("\nDecision Tree:")
print(classification_report(y_test, y_pred_dt))
print(f"Accuracy: {accuracy_dt}")

In [ ]:
X_test22=X_test[selected_features]

In [ ]:
# List of features you want to use
selected_features = ['J90','E87.6','R06.02']  # replace with your actual feature names

X = df10[selected_features]
y = df10['dialysis']

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

X_train, X_test4, y_train, y_test4 = train_test_split(X, y, test_size=0.2, random_state=42)

classifier = LogisticRegression()
classifier.fit(X_train, y_train)

# Evaluate the trained model
y_pred = classifier.predict(X_test22)
accuracy = accuracy_score(y_test, y_pred)
print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy}")


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
ConfusionMatrixDisplay.from_estimator(
    classifier, X_test, y_test, xticks_rotation="vertical"
)
plt.tight_layout()
plt.show()

In [ ]:
df['X82248'].mean()

In [ ]:
df10['X82248'].mean()

In [ ]:
generated_df['X82248'].mean()

In [ ]:
df['X82565'].mean()

In [ ]:
df10['X82565'].mean()

In [ ]:
generated_df['X82565'].mean()

In [ ]:
both_ones_count = len(df[(df['X82248'] == 1) & (df['X82565'] == 1)])

# Calculate the percentage
percentage = (both_ones_count / len(df)) * 100

# Print the result
print("Percentage of occurrences where both features are 1: {:.2f}%".format(percentage))

In [ ]:
both_ones_count = len(generated_df[(generated_df['X82248'] == 1) & (generated_df['X82565'] == 1)])

# Calculate the percentage
percentage = (both_ones_count / len(generated_df)) * 100

# Print the result
print("Percentage of occurrences where both features are 1: {:.2f}%".format(percentage))

In [ ]:
both_ones_count = len(df10[(df10['X82248'] == 1) & (df10['X82565'] == 1)])

# Calculate the percentage
percentage = (both_ones_count / len(df10)) * 100

# Print the result
print("Percentage of occurrences where both features are 1: {:.2f}%".format(percentage))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
# For df1
percentage_X82248_df1 = df['X82248'].mean() * 100
percentage_X82565_df1 = df['X82565'].mean() * 100
percentage_both_df1 = df[(df['X82248'] == 1) & (df['X82565'] == 1)].shape[0] / df.shape[0] * 100

# For df2
percentage_X82248_df2 = df10['X82248'].mean() * 100
percentage_X82565_df2 = df10['X82565'].mean() * 100
percentage_both_df2 = df10[(df10['X82248'] == 1) & (df10['X82565'] == 1)].shape[0] / df10.shape[0] * 100

# Data to plot
labels = ['X82248', 'X82565', 'Both']
percentages_df1 = [percentage_X82248_df1, percentage_X82565_df1, percentage_both_df1]
percentages_df2 = [percentage_X82248_df2, percentage_X82565_df2, percentage_both_df2]

# X-axis locations for groups
x = np.arange(len(labels))

# Width of the bars
width = 0.35

# Create grouped bar plot
fig, ax = plt.subplots()
rects1 = ax.bar(x - width/2, percentages_df1, width, label='Real Data')
rects2 = ax.bar(x + width/2, percentages_df2, width, label='Final Data')

# Adding title and labels
ax.set_title('Distribution of Features for Two Datasets')
ax.set_xlabel('Features')
ax.set_ylabel('Percentage (%)')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

# Show the plot
plt.show()


In [ ]:
import plotly.graph_objects as go

categories = ['Accuracy','Percision','Recall', 'F1-score']

fig = go.Figure()

fig.add_trace(go.Scatterpolar(
      r=[0.93,0.93, 0.94, 0.94],
      theta=categories,
      fill='toself',
      name='GAN'
))

fig.add_trace(go.Scatterpolar(
      r=[0.87, 0.88, 0.91, 0.90],
      theta=categories,
      fill='toself',
      name='Smote'
))
fig.add_trace(go.Scatterpolar(
      r=[0.94, 0.96, 0.98, 0.97],
      theta=categories,
      fill='toself',
      name='Real Data'
))


fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0.3, 1]
        ),
        angularaxis=dict(
            direction="clockwise",
            rotation=45
        )
    ),
    showlegend=True,
    legend=dict(
        x=0.429,
        y=1.05
    )
)

fig.show()


In [ ]:
import plotly.graph_objects as go

categories = ['Accuracy','Percision','Recall', 'F1-score']

fig = go.Figure()

fig.add_trace(go.Scatterpolar(
      r=[0.93,0.92, 0.90, 0.91],
      theta=categories,
      fill='toself',
      name='GAN'
))

fig.add_trace(go.Scatterpolar(
      r=[0.94, 0.69, 0.26, 0.57],
      theta=categories,
      fill='toself',
      name='Real Data'
))


fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0.3, 1]
        ),
        angularaxis=dict(
            direction="clockwise",
            rotation=45
        )
    ),
    showlegend=True,
    legend=dict(
        x=0.429,
        y=1.05
    )
)

fig.show()


In [ ]:
percentage_X82248 = df['X82248'].mean() * 100
percentage_X82565 = df['X82565'].mean() * 100
percentage_both = df[(df['X82248'] == 1) & (df['X82565'] == 1)].shape[0] / df.shape[0] * 100

print(f"Percentage of ones in X82248: {percentage_X82248:.2f}%")
print(f"Percentage of ones in X82565: {percentage_X82565:.2f}%")
print(f"Percentage of ones in both features: {percentage_both:.2f}%")


In [ ]:
import matplotlib.pyplot as plt

# Data to plot
labels = ['X82248', 'X82565', 'Both']
percentages = [percentage_X82248, percentage_X82565, percentage_both]

# Create bar plot
plt.bar(labels, percentages)

# Adding title and labels
plt.title('Percentage of Ones in Features')
plt.xlabel('Features')
plt.ylabel('Percentage (%)')

# Show the plot
plt.show()


In [ ]:
percentage_X82248 = generated_df['X82248'].mean() * 100
percentage_X82565 = generated_df['X82565'].mean() * 100
percentage_both = generated_df[(generated_df['X82248'] == 1) & (generated_df['X82565'] == 1)].shape[0] / generated_df.shape[0] * 100

print(f"Percentage of ones in X82248: {percentage_X82248:.2f}%")
print(f"Percentage of ones in X82565: {percentage_X82565:.2f}%")
print(f"Percentage of ones in both features: {percentage_both:.2f}%")


In [ ]:
import matplotlib.pyplot as plt

# Data to plot
labels = ['X82248', 'X82565', 'Both']
percentages = [percentage_X82248, percentage_X82565, percentage_both]

# Create bar plot
plt.bar(labels, percentages)

# Adding title and labels
plt.title('Percentage of Ones in Features')
plt.xlabel('Features')
plt.ylabel('Percentage (%)')

# Show the plot
plt.show()


In [ ]:
!pip install table_evaluator

In [ ]:
from table_evaluator import load_data, TableEvaluator

In [ ]:
table_evaluator = TableEvaluator(df, generated_df)

In [ ]:
table_evaluator.visual_evaluation()

In [ ]:

# Split your original data into training and test sets
X_real = df.drop('dialysis', axis=1)
y_real = df['dialysis']

X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(X_real, y_real, test_size=0.2, random_state=42)
X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(X_real, y_real, test_size=0.2, random_state=42)

# Add the generated data to the training data
X_train = pd.concat([X_train_real, generated_df.drop(['dialysis'],1)])
y_train = pd.concat([y_train_real, pd.Series([1]*len(generated_df))])

In [ ]:
# Logistic Regression Classifier
X_train, X_test, y_train, y_test = train_test_split(balanced_X, balanced_y, test_size=0.2, random_state=42)
classifier = LogisticRegression()
classifier.fit(X_train, y_train)

# Evaluate the classifier
y_pred = classifier.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LeakyReLU
from tensorflow.keras.optimizers import Adam



X = df.drop('dialysis', axis=1)
y = df['dialysis']


# Step 2: Split the dataset into majority and minority classes
majority_class = X[y == 0]
minority_class = X[y == 1]


# Step 3: Define and train the GAN model
def define_generator(latent_dim, n_outputs):
    model = Sequential()
    model.add(Dense(10, input_dim=latent_dim))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(n_outputs, activation='sigmoid'))
    return model


def define_discriminator(n_inputs):
    model = Sequential()
    model.add(Dense(25, input_dim=n_inputs))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer=Adam(lr=0.001), metrics=['accuracy'])
    return model


def define_gan(generator, discriminator):
    discriminator.trainable = False
    model = Sequential()
    model.add(generator)
    model.add(discriminator)
    model.compile(loss='binary_crossentropy', optimizer=Adam(lr=0.001))
    return model


def generate_real_samples(dataset, n_samples):
    ix = np.random.randint(0, dataset.shape[0], n_samples)
    X = dataset.iloc[ix]  # Use iloc instead of indexing directly
    y = np.ones((n_samples, 1))
    return X, y


def generate_latent_points(latent_dim, n_samples):
    x_input = np.random.randn(latent_dim * n_samples)
    x_input = x_input.reshape(n_samples, latent_dim)
    return x_input


def generate_fake_samples(generator, latent_dim, n_samples):
    x_input = generate_latent_points(latent_dim, n_samples)
    X = generator.predict(x_input)
    y = np.zeros((n_samples, 1))
    return X, y


def train_gan(generator, discriminator, gan_model, latent_dim, n_epochs, n_batch, majority_data):
    half_batch = int(n_batch / 2)
    g_losses = []
    d_losses = []
    for epoch in range(n_epochs):
        X_real, y_real = generate_real_samples(majority_data, half_batch)
        d_loss1, _ = discriminator.train_on_batch(X_real, y_real)
        X_fake, y_fake = generate_fake_samples(generator, latent_dim, half_batch)
        d_loss2, _ = discriminator.train_on_batch(X_fake, y_fake)
        X_gan = generate_latent_points(latent_dim, n_batch)
        y_gan = np.ones((n_batch, 1))
        g_loss = gan_model.train_on_batch(X_gan, y_gan)
        print(f'Epoch: {epoch+1}, Discriminator Loss: {0.5 * (d_loss1 + d_loss2)}, Generator Loss: {g_loss}')
        g_losses.append(g_loss)
        d_losses.append(0.5 * (d_loss1 + d_loss2))

    # Plot losses
    plt.plot(g_losses, label='Generator Loss')
    plt.plot(d_losses, label='Discriminator Loss')
    plt.title('GAN Training Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()


# Set hyperparameters
latent_dim = 10
n_epochs = 1000
n_batch = 128

# Define and compile the discriminator
discriminator = define_discriminator(minority_class.shape[1])
# Define and compile the generator
generator = define_generator(latent_dim, minority_class.shape[1])
# Define and compile the GAN model
gan_model = define_gan(generator, discriminator)

# Train the GAN model and plot losses
train_gan(generator, discriminator, gan_model, latent_dim, n_epochs, n_batch, majority_class)


# Step 4: Generate synthetic samples
n_samples =3074
latent_points = generate_latent_points(latent_dim, n_samples)
synthetic_samples = generator.predict(latent_points)

feature_names = ['X82248', 'X82565']
desired_means = [0.59, 0.49]
adjustments = []
for feature_name, desired_mean in zip(feature_names, desired_means):
    column_index = X.columns.get_loc(feature_name)
    column = synthetic_samples[:, column_index]
    current_mean = column.mean()
    adjustment = desired_mean - current_mean
    adjustments.append(adjustment)

# Apply the adjustments to the generated data
for i, feature_name in enumerate(feature_names):
    column_index = X.columns.get_loc(feature_name)
    synthetic_samples[:, column_index] += adjustments[i]

# Generated data (synthetic samples)
generated_data = synthetic_samples.copy()

# Step 5: Combine synthetic samples with the original minority class
combined_samples = np.concatenate((minority_class, synthetic_samples))


# Step 6: Combine minority and majority class samples to create a balanced dataset
n_majority_samples = combined_samples.shape[0]
n_samples_per_class = n_majority_samples // 2

# Shuffle the combined samples
np.random.shuffle(combined_samples)

balanced_X = np.concatenate((combined_samples[:n_samples_per_class], majority_class.sample(n_samples_per_class)))
balanced_y = np.concatenate((np.ones(n_samples_per_class), np.zeros(n_samples_per_class)))

# Step 7: Split the balanced dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(balanced_X, balanced_y, test_size=0.2, random_state=42)

# Split the training and testing sets for each class
X_train_minority = X_train[y_train == 1]
X_train_majority = X_train[y_train == 0]
X_test_minority = X_test[y_test == 1]
X_test_majority = X_test[y_test == 0]

# Randomly select equal number of samples from each class for training and testing
np.random.shuffle(X_train_minority)
np.random.shuffle(X_train_majority)
np.random.shuffle(X_test_minority)
np.random.shuffle(X_test_majority)

n_train_samples = min(X_train_minority.shape[0], X_train_majority.shape[0])
n_test_samples = min(X_test_minority.shape[0], X_test_majority.shape[0])

X_train = np.concatenate((X_train_minority[:n_train_samples], X_train_majority[:n_train_samples]))
y_train = np.concatenate((np.ones(n_train_samples), np.zeros(n_train_samples)))
X_test = np.concatenate((X_test_minority[:n_test_samples], X_test_majority[:n_test_samples]))
y_test = np.concatenate((np.ones(n_test_samples), np.zeros(n_test_samples)))

# Step 8: Train a classification model on the balanced dataset
classifier = LogisticRegression()
classifier.fit(X_train, y_train)

# Step 9: Evaluate the trained model
y_pred = classifier.predict(X_test)
print(classification_report(y_test, y_pred))









In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LeakyReLU
from tensorflow.keras.optimizers import Adam

# Your data preparation step here
# I am assuming that you have a dataframe df with 'dialysis' as the target variable
X = df.drop('dialysis', axis=1)
y = df['dialysis']

majority_class = X[y == 0]
minority_class = X[y == 1]

# GAN components
def define_generator(latent_dim, n_outputs):
    model = Sequential()
    model.add(Dense(10, input_dim=latent_dim))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(n_outputs, activation='sigmoid'))
    return model

def define_discriminator(n_inputs):
    model = Sequential()
    model.add(Dense(25, input_dim=n_inputs))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer=Adam(lr=0.001), metrics=['accuracy'])
    return model

def define_gan(generator, discriminator):
    discriminator.trainable = False
    model = Sequential()
    model.add(generator)
    model.add(discriminator)
    model.compile(loss='binary_crossentropy', optimizer=Adam(lr=0.001))
    return model

# data for GAN
def generate_real_samples(dataset, n_samples):
    ix = np.random.randint(0, dataset.shape[0], n_samples)
    X = dataset.iloc[ix]
    y = np.ones((n_samples, 1))
    return X, y

def generate_latent_points(latent_dim, n_samples):
    x_input = np.random.randn(latent_dim * n_samples)
    x_input = x_input.reshape(n_samples, latent_dim)
    return x_input

def generate_fake_samples(generator, latent_dim, n_samples):
    x_input = generate_latent_points(latent_dim, n_samples)
    X = generator.predict(x_input)
    y = np.zeros((n_samples, 1))
    return X, y

# GAN training
def train_gan(generator, discriminator, gan_model, latent_dim, n_epochs, n_batch, majority_data):
    half_batch = int(n_batch / 2)
    g_losses = []
    d_losses = []
    real_ratio = 0.5
    fake_ratio = 1 - real_ratio
    for epoch in range(n_epochs):
        for _ in range(2):
            # use int() to ensure the result is an integer
            X_real, y_real = generate_real_samples(majority_data, int(half_batch * real_ratio))
            X_fake, y_fake = generate_fake_samples(generator, latent_dim, int(half_batch * fake_ratio))
            d_loss_real, _ = discriminator.train_on_batch(X_real, y_real)
            d_loss_fake, _ = discriminator.train_on_batch(X_fake, y_fake)
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

        X_gan = generate_latent_points(latent_dim, n_batch)
        y_gan = np.ones((n_batch, 1))
        g_loss = gan_model.train_on_batch(X_gan, y_gan)
        print(f'Epoch: {epoch+1}, Discriminator Loss: {d_loss}, Generator Loss: {g_loss}')
        g_losses.append(g_loss)
        d_losses.append(d_loss)

    plt.plot(g_losses, label='Generator Loss')
    plt.plot(d_losses, label='Discriminator Loss')
    plt.title('GAN Training Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()


# Set hyperparameters
latent_dim = 5
n_epochs = 400
n_batch = 63

discriminator = define_discriminator(majority_class.shape[1])
generator = define_generator(latent_dim, majority_class.shape[1])
gan_model = define_gan(generator, discriminator)

# Train the GAN
train_gan(generator, discriminator, gan_model, latent_dim, n_epochs, n_batch, majority_class)

# Generate new minority samples
n_samples = 3000
latent_points = generate_latent_points(latent_dim, n_samples)
synthetic_samples = generator.predict(latent_points)

# Combine with existing minority samples
combined_samples = np.concatenate((minority_class, synthetic_samples))

# Combine with majority samples for balanced dataset
n_majority_samples = combined_samples.shape[0]
balanced_X = np.concatenate((combined_samples, majority_class.sample(n_majority_samples)))
balanced_y = np.concatenate((np.ones(n_majority_samples), np.zeros(n_majority_samples)))

# Logistic Regression Classifier
X_train, X_test, y_train, y_test = train_test_split(balanced_X, balanced_y, test_size=0.2, random_state=42)
classifier = LogisticRegression()
classifier.fit(X_train, y_train)

# Evaluate the classifier
y_pred = classifier.predict(X_test)
print(classification_report(y_test, y_pred))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LeakyReLU
from tensorflow.keras.optimizers import Adam



X = df.drop('dialysis', axis=1)
y = df['dialysis']


# Step 2: Split the dataset into majority and minority classes
majority_class = X[y == 0]
minority_class = X[y == 1]


In [ ]:
def define_generator(latent_dim, n_outputs):
    model = Sequential()
    model.add(Dense(10, input_dim=latent_dim))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(n_outputs, activation='sigmoid'))
    return model


def define_discriminator(n_inputs):
    model = Sequential()
    model.add(Dense(25, input_dim=n_inputs))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer=Adam(lr=0.001), metrics=['accuracy'])
    return model


def define_gan(generator, discriminator):
    discriminator.trainable = False
    model = Sequential()
    model.add(generator)
    model.add(discriminator)
    model.compile(loss='binary_crossentropy', optimizer=Adam(lr=0.001))
    return model


def generate_real_samples(dataset, n_samples):
    ix = np.random.randint(0, dataset.shape[0], n_samples)
    X = dataset.iloc[ix]  # Use iloc instead of indexing directly
    y = np.ones((n_samples, 1))
    return X, y


def generate_latent_points(latent_dim, n_samples):
    x_input = np.random.randn(latent_dim * n_samples)
    x_input = x_input.reshape(n_samples, latent_dim)
    return x_input


def generate_fake_samples(generator, latent_dim, n_samples):
    x_input = generate_latent_points(latent_dim, n_samples)
    X = generator.predict(x_input)
    y = np.zeros((n_samples, 1))
    return X, y

def train_gan(generator, discriminator, gan_model, latent_dim, n_epochs, n_batch, majority_data):
    half_batch = int(n_batch / 2)
    g_losses = []
    d_losses = []
    # manually specify the ratio of real to fake samples
    real_ratio = 0.5
    fake_ratio = 1 - real_ratio
    for epoch in range(n_epochs):
        # update discriminator more often than generator
        for _ in range(2):
            X_real, y_real = generate_real_samples(majority_data, half_batch * real_ratio)
            X_fake, y_fake = generate_fake_samples(generator, latent_dim, half_batch * fake_ratio)
            d_loss_real, _ = discriminator.train_on_batch(X_real, y_real)
            d_loss_fake, _ = discriminator.train_on_batch(X_fake, y_fake)
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

        # prepare points in latent space as input for the generator
        x_gan = generate_latent_points(latent_dim, n_batch)
        y_gan = np.ones((n_batch, 1))
        # update the generator via the discriminator's error
        g_loss = gan_model.train_on_batch(x_gan, y_gan)

        print(f'Epoch: {epoch+1}, Discriminator Loss: {d_loss}, Generator Loss: {g_loss}')
        g_losses.append(g_loss)
        d_losses.append(d_loss)

    # Plot losses
    plt.plot(g_losses, label='Generator Loss')
    plt.plot(d_losses, label='Discriminator Loss')
    plt.title('GAN Training Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()


In [ ]:
latent_dim = 5
n_epochs = 400
n_batch = 63

# Define and compile the discriminator
discriminator = define_discriminator(minority_class.shape[1])
# Define and compile the generator
generator = define_generator(latent_dim, minority_class.shape[1])
# Define and compile the GAN model
gan_model = define_gan(generator, discriminator)

# Train the GAN model and plot losses
train_gan(generator, discriminator, gan_model, latent_dim, n_epochs, n_batch, majority_class)


# Step 4: Generate synthetic samples
n_samples =3000
latent_points = generate_latent_points(latent_dim, n_samples)
synthetic_samples = generator.predict(latent_points)


# Generated data (synthetic samples)
generated_data = synthetic_samples.copy()


# Step 5: Combine synthetic samples with the original minority class
combined_samples = np.concatenate((minority_class, synthetic_samples))

# Step 6: Combine minority and majority class samples to create a balanced dataset
n_majority_samples = combined_samples.shape[0]
balanced_X = np.concatenate((combined_samples, majority_class.sample(n_majority_samples)))
balanced_y = np.concatenate((np.ones(n_majority_samples), np.zeros(n_majority_samples)))

# Step 7: Train a classification model on the balanced dataset
X_train, X_test, y_train, y_test = train_test_split(balanced_X, balanced_y, test_size=0.2, random_state=42)

classifier = LogisticRegression()
classifier.fit(X_train, y_train)

# Step 8: Evaluate the trained model
y_pred = classifier.predict(X_test)
print(classification_report(y_test, y_pred))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LeakyReLU
from tensorflow.keras.optimizers import Adam



X = df.drop('dialysis', axis=1)
y = df['dialysis']


# Step 2: Split the dataset into majority and minority classes
majority_class = X[y == 0]
minority_class = X[y == 1]

from tensorflow.keras.layers import Dropout, BatchNormalization

def define_generator(latent_dim, n_outputs):
    model = Sequential()
    model.add(Dense(30, input_dim=latent_dim))  # Increase the number of neurons
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization())  # Add BatchNormalization layer
    model.add(Dense(20))  # Add additional Dense layer
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dropout(0.5))  # Add Dropout layer
    model.add(Dense(n_outputs, activation='sigmoid'))
    return model


def define_discriminator(n_inputs):
    model = Sequential()
    model.add(Dense(25, input_dim=n_inputs))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer=Adam(lr=0.0005), metrics=['accuracy'])  # Decreased learning rate
    return model

def define_gan(generator, discriminator):
    discriminator.trainable = False
    model = Sequential()
    model.add(generator)
    model.add(discriminator)
    model.compile(loss='binary_crossentropy', optimizer=Adam(lr=0.0015))  # Increased learning rate
    return model

def generate_real_samples(dataset, n_samples):
    ix = np.random.randint(0, dataset.shape[0], n_samples)
    X = dataset.iloc[ix]  # Use iloc instead of indexing directly
    y = np.ones((n_samples, 1))
    return X, y


def generate_latent_points(latent_dim, n_samples):
    x_input = np.random.randn(latent_dim * n_samples)
    x_input = x_input.reshape(n_samples, latent_dim)
    return x_input


def generate_fake_samples(generator, latent_dim, n_samples):
    x_input = generate_latent_points(latent_dim, n_samples)
    X = generator.predict(x_input)
    y = np.zeros((n_samples, 1))
    return X, y


def train_gan(generator, discriminator, gan_model, latent_dim, n_epochs, n_batch, majority_data):
    half_batch = int(n_batch / 2)
    g_losses = []
    d_losses = []
    for epoch in range(n_epochs):
        X_real, y_real = generate_real_samples(majority_data, half_batch)
        d_loss1, _ = discriminator.train_on_batch(X_real, y_real)
        X_fake, y_fake = generate_fake_samples(generator, latent_dim, half_batch)
        d_loss2, _ = discriminator.train_on_batch(X_fake, y_fake)
        X_gan = generate_latent_points(latent_dim, n_batch)
        y_gan = np.ones((n_batch, 1))
        g_loss = gan_model.train_on_batch(X_gan, y_gan)
        print(f'Epoch: {epoch+1}, Discriminator Loss: {0.5 * (d_loss1 + d_loss2)}, Generator Loss: {g_loss}')
        g_losses.append(g_loss)
        d_losses.append(0.5 * (d_loss1 + d_loss2))

    # Plot losses
    plt.plot(g_losses, label='Generator Loss')
    plt.plot(d_losses, label='Discriminator Loss')
    plt.title('GAN Training Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()


# Set hyperparameters
latent_dim = 5
n_epochs = 400
n_batch = 63

# Define and compile the discriminator
discriminator = define_discriminator(minority_class.shape[1])
# Define and compile the generator
generator = define_generator(latent_dim, minority_class.shape[1])
# Define and compile the GAN model
gan_model = define_gan(generator, discriminator)

# Train the GAN model and plot losses
train_gan(generator, discriminator, gan_model, latent_dim, n_epochs, n_batch, majority_class)


# Step 4: Generate synthetic samples
n_samples =3000
latent_points = generate_latent_points(latent_dim, n_samples)
synthetic_samples = generator.predict(latent_points)


# Generated data (synthetic samples)
generated_data = synthetic_samples.copy()


# Step 5: Combine synthetic samples with the original minority class
combined_samples = np.concatenate((minority_class, synthetic_samples))

# Step 6: Combine minority and majority class samples to create a balanced dataset
n_majority_samples = combined_samples.shape[0]
balanced_X = np.concatenate((combined_samples, majority_class.sample(n_majority_samples)))
balanced_y = np.concatenate((np.ones(n_majority_samples), np.zeros(n_majority_samples)))

# Step 7: Train a classification model on the balanced dataset
X_train, X_test, y_train, y_test = train_test_split(balanced_X, balanced_y, test_size=0.2, random_state=42)

classifier = LogisticRegression()
classifier.fit(X_train, y_train)

# Step 8: Evaluate the trained model
y_pred = classifier.predict(X_test)
print(classification_report(y_test, y_pred))


In [ ]:
import sdv

In [ ]:
import sdv
print(sdv.__version__)


In [ ]:
!pip install ctgan

In [ ]:
df20=df[df['dialysis']==1]

In [ ]:
from ctgan import CTGAN
from ctgan import load_demo


# Names of the columns that are discrete
discrete_columns = df[selected_features]

ctgan = CTGAN(epochs=10)
ctgan.fit(df20, discrete_columns)

# Create synthetic data
synthetic_data = ctgan.sample(3000)
import pandas as pd

# Assuming you have two DataFrames df1 and df2 with the same columns
# Here's how you can merge them

# Concatenate the DataFrames vertically (along rows) to create a new DataFrame with both datasets combined
df30 = pd.concat([df,synthetic_data ], ignore_index=True)

In [ ]:
synthetic_data['dialysis'].value_counts()

In [ ]:
import pandas as pd

# Assuming you have two DataFrames df1 and df2 with the same columns
# Here's how you can merge them

# Concatenate the DataFrames vertically (along rows) to create a new DataFrame with both datasets combined
df30 = pd.concat([df,synthetic_data ], ignore_index=True)



In [ ]:
generated_data

In [ ]:
generated_df = pd.DataFrame(data=generated_data, columns=minority_class.columns)

In [ ]:
generated_df

In [ ]:
# Calculate the mean of each column in the generated DataFrame
column_means = generated_df.mean()

# Iterate over each column in the generated DataFrame and set values greater than the mean to 1 and others to 0
for column in generated_df.columns:
    generated_df[column] = (generated_df[column] > column_means[column]).astype(int)




In [ ]:
generated_df['dialysis'] = 1
generated_df

In [ ]:
import pandas as pd

# Assuming you have two DataFrames df1 and df2 with the same columns
# Here's how you can merge them

# Concatenate the DataFrames vertically (along rows) to create a new DataFrame with both datasets combined
df3 = pd.concat([df, generated_df], ignore_index=True)



In [ ]:
df3

In [ ]:
X = df3.drop('dialysis', axis=1)  # Replace 'dialysis' with the actual target column name
y = df3['dialysis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)


classifier = LogisticRegression()
classifier.fit(X_train, y_train)

# Step 8: Evaluate the trained model
y_pred = classifier.predict(X_test)
print(classification_report(y_test, y_pred))


In [ ]:
from sklearn.model_selection import train_test_split

# Split your original data into training and test sets
X_real = df.drop('dialysis', axis=1)
y_real = df['dialysis']

X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(X_real, y_real, test_size=0.2, random_state=42)
X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(X_real, y_real, test_size=0.2, random_state=42)

# Add the generated data to the training data
X_train = pd.concat([X_train_real, generated_df.drop(['dialysis'],1)])
y_train = pd.concat([y_train_real, pd.Series([1]*len(generated_df))])
# Now, you can train your model on X_train and y_train, and evaluate it on X_test_real and y_test_real

classifier = LogisticRegression()
classifier.fit(X_train, y_train)

# Step 8: Evaluate the trained model
y_pred = classifier.predict(X_test)
print(classification_report(y_test, y_pred))


In [ ]:
df.shape

In [ ]:
pd.Series([1]*len(generated_df))

In [ ]:
minority_X_train_real['dialysis'].isna().sum()

In [ ]:
X_train

In [ ]:
pip install table_evaluator

In [ ]:
from table_evaluator import load_data, TableEvaluator

In [ ]:
table_evaluator = TableEvaluator(df, generated_df)

In [ ]:
table_evaluator.visual_evaluation()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt



# Compute the correlation matrix
corr = df.corr()

# Generate a mask for the upper triangle
mask = np.triu(np.ones_like(corr, dtype=bool))

# Set up the matplotlib figure
f, ax = plt.subplots(figsize=(11, 9))

# Generate a custom diverging colormap
cmap = sns.diverging_palette(230, 20, as_cmap=True)

# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(corr, mask=mask, cmap=cmap, vmax=.3, center=0,
            square=True, linewidths=.5, cbar_kws={"shrink": .5})

plt.title('Correlation heatmap')
plt.show()


In [ ]:
# Pairplot
sns.pairplot(df, hue='dialysis')
plt.title('Pairwise plots of all features')
plt.show()


In [ ]:
df['code_82565'].mean()

In [ ]:
generated_df['code_82565'].mean()

In [ ]:
df3['code_82565'].mean()

In [ ]:
df['dialysis'].value_counts()

In [ ]:
balanced_X.shape

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split

# Set up the data
X = df.drop('dialysis', axis=1)
y = df['dialysis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the classifier
classifier = DecisionTreeClassifier(random_state=42)
classifier.fit(X_train, y_train)

# Evaluate the trained model
y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy}")


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split

# Set up the data
X = df30.drop('dialysis', axis=1)
y = df30['dialysis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the classifier
classifier = DecisionTreeClassifier(random_state=42)
classifier.fit(X_train, y_train)

# Evaluate the trained model
y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy}")


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Set up the data
X = df.drop('dialysis', axis=1)
y = df['dialysis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the classifier
classifier = RandomForestClassifier(random_state=42)
classifier.fit(X_train, y_train)

# Evaluate the trained model
y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy}")


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Set up the data
X = df3.drop('dialysis', axis=1)
y = df3['dialysis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the classifier
classifier = RandomForestClassifier(random_state=42)
classifier.fit(X_train, y_train)

# Evaluate the trained model
y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy}")


In [ ]:
from sklearn import svm

# Set up the data
X = df.drop('dialysis', axis=1)
y = df['dialysis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the classifier
classifier = svm.SVC(random_state=42)
classifier.fit(X_train, y_train)

# Evaluate the trained model
y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy}")


In [ ]:
from sklearn import svm

# Set up the data
X = df3.drop('dialysis', axis=1)
y = df3['dialysis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the classifier
classifier = svm.SVC(random_state=42)
classifier.fit(X_train, y_train)

# Evaluate the trained model
y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy}")


In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Set up the data
X = df.drop('dialysis', axis=1)
y = df['dialysis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the classifier with max_depth = 3
classifier = DecisionTreeClassifier(max_depth=3, random_state=42)
classifier.fit(X_train, y_train)

# Plot the decision tree
plt.figure(figsize=(15,10))  # Set the figure size (optional)
plot_tree(classifier,
          filled=True,
          rounded=True,
          class_names=["No Dialysis", "Dialysis"],
          feature_names=X.columns)
plt.show()


In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Set up the data
X = df3.drop('dialysis', axis=1)
y = df3['dialysis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the classifier with max_depth = 3
classifier = DecisionTreeClassifier(max_depth=3, random_state=42)
classifier.fit(X_train, y_train)

# Plot the decision tree
plt.figure(figsize=(15,10))  # Set the figure size (optional)
plot_tree(classifier,
          filled=True,
          rounded=True,
          class_names=["No Dialysis", "Dialysis"],
          feature_names=X.columns)
plt.show()


In [ ]:
df['code_85025'].mean()

In [ ]:
generated_df['code_85025'].mean()

In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import io
df = pd.read_csv(io.BytesIO(uploaded['C1forDialysisPrediction.csv']))

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df=df.drop(['Unnamed: 0', 'patient_id','CKDdate','dialysisdate'],1)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LeakyReLU
from tensorflow.keras.optimizers import Adam



X = df.drop('dialysis', axis=1)
y = df['dialysis']


# Step 2: Split the dataset into majority and minority classes
majority_class = X[y == 0]
minority_class = X[y == 1]


# Step 3: Define and train the GAN model
from tensorflow.keras.layers import Dropout, BatchNormalization

def define_generator(latent_dim, n_outputs):
    model = Sequential()
    model.add(Dense(100, input_dim=latent_dim))
    model.add(BatchNormalization())
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(50))
    model.add(BatchNormalization())
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(30))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(n_outputs, activation='sigmoid'))
    return model

def define_discriminator(n_inputs):
    model = Sequential()
    model.add(Dense(60, input_dim=n_inputs))
    model.add(Dropout(0.4))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(30))
    model.add(Dropout(0.4))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(15))
    model.add(Dropout(0.4))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer=Adam(lr=0.001), metrics=['accuracy'])
    return model



def define_gan(generator, discriminator):
    discriminator.trainable = False
    model = Sequential()
    model.add(generator)
    model.add(discriminator)
    model.compile(loss='binary_crossentropy', optimizer=Adam(lr=0.001))
    return model


def generate_real_samples(dataset, n_samples):
    ix = np.random.randint(0, dataset.shape[0], n_samples)
    X = dataset.iloc[ix]  # Use iloc instead of indexing directly
    y = np.ones((n_samples, 1))
    return X, y


def generate_latent_points(latent_dim, n_samples):
    x_input = np.random.randn(latent_dim * n_samples)
    x_input = x_input.reshape(n_samples, latent_dim)
    return x_input


def generate_fake_samples(generator, latent_dim, n_samples):
    x_input = generate_latent_points(latent_dim, n_samples)
    X = generator.predict(x_input)
    y = np.zeros((n_samples, 1))
    return X, y


def train_gan(generator, discriminator, gan_model, latent_dim, n_epochs, n_batch, majority_data):
    half_batch = int(n_batch / 2)
    g_losses = []
    d_losses = []
    for epoch in range(n_epochs):
        X_real, y_real = generate_real_samples(majority_data, half_batch)
        d_loss1, _ = discriminator.train_on_batch(X_real, y_real)
        X_fake, y_fake = generate_fake_samples(generator, latent_dim, half_batch)
        d_loss2, _ = discriminator.train_on_batch(X_fake, y_fake)
        X_gan = generate_latent_points(latent_dim, n_batch)
        y_gan = np.ones((n_batch, 1))
        g_loss = gan_model.train_on_batch(X_gan, y_gan)
        print(f'Epoch: {epoch+1}, Discriminator Loss: {0.5 * (d_loss1 + d_loss2)}, Generator Loss: {g_loss}')
        g_losses.append(g_loss)
        d_losses.append(0.5 * (d_loss1 + d_loss2))

    # Plot losses
    plt.plot(g_losses, label='Generator Loss')
    plt.plot(d_losses, label='Discriminator Loss')
    plt.title('GAN Training Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()


# Set hyperparameters
latent_dim = 5
n_epochs = 100
n_batch = 63

# Define and compile the discriminator
discriminator = define_discriminator(minority_class.shape[1])
# Define and compile the generator
generator = define_generator(latent_dim, minority_class.shape[1])
# Define and compile the GAN model
gan_model = define_gan(generator, discriminator)

# Train the GAN model and plot losses
train_gan(generator, discriminator, gan_model, latent_dim, n_epochs, n_batch, majority_class)


# Step 4: Generate synthetic samples
n_samples =3000
latent_points = generate_latent_points(latent_dim, n_samples)
synthetic_samples = generator.predict(latent_points)


# Generated data (synthetic samples)
generated_data = synthetic_samples.copy()


# Step 5: Combine synthetic samples with the original minority class
combined_samples = np.concatenate((minority_class, synthetic_samples))

# Step 6: Combine minority and majority class samples to create a balanced dataset
n_majority_samples = combined_samples.shape[0]
balanced_X = np.concatenate((combined_samples, majority_class.sample(n_majority_samples)))
balanced_y = np.concatenate((np.ones(n_majority_samples), np.zeros(n_majority_samples)))

# Step 7: Train a classification model on the balanced dataset
X_train, X_test, y_train, y_test = train_test_split(balanced_X, balanced_y, test_size=0.2, random_state=42)

classifier = LogisticRegression()
classifier.fit(X_train, y_train)

# Step 8: Evaluate the trained model
y_pred = classifier.predict(X_test)
print(classification_report(y_test, y_pred))


In [ ]:
X = df3.drop('dialysis', axis=1)  # Replace 'dialysis' with the actual target column name
y = df3['dialysis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)


classifier = LogisticRegression()
classifier.fit(X_train, y_train)

# Step 8: Evaluate the trained model
y_pred = classifier.predict(X_test)
print(classification_report(y_test, y_pred))


In [ ]:
X = df.drop('dialysis', axis=1)  # Replace 'dialysis' with the actual target column name
y = df['dialysis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)


classifier = LogisticRegression()
classifier.fit(X_train, y_train)

# Step 8: Evaluate the trained model
y_pred = classifier.predict(X_test)
print(classification_report(y_test, y_pred))


In [ ]:
generated_df

In [ ]:
from sklearn.model_selection import train_test_split

# Split your original data into training and test sets
X_real = df.drop('dialysis', axis=1)
y_real = df['dialysis']

X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(X_real, y_real, test_size=0.2, random_state=42)
X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(X_real, y_real, test_size=0.2, random_state=42)

# Add the generated data to the training data
X_train = pd.concat([X_train_real, generated_df.drop(['dialysis'],1)])
y_train = pd.concat([y_train_real, pd.Series([1]*len(generated_df))])
# Now, you can train your model on X_train and y_train, and evaluate it on X_test_real and y_test_real

classifier = LogisticRegression()
classifier.fit(X_train, y_train)

# Step 8: Evaluate the trained model
y_pred = classifier.predict(X_test)
print(classification_report(y_test, y_pred))
